# Cell 1 — Environment Setup and Health Check

In [ ]:
# ============================================================
# Cell 1 — Environment Setup and Health Check
# ------------------------------------------------------------
# This cell installs and verifies only the libraries required
# for the NLP component of the MSc AI Dissertation project.
# The function below performs safe installation, avoiding
# unnecessary reinstallation when a package already exists.
# ============================================================

import sys, subprocess, importlib, platform, random, os
import pkgutil

def safe_pip_install(pkg_spec, import_name=None):
    """
    Attempt to import a package; if missing, install it quietly.
    If the quiet installation fails, retry with verbose output.
    Example:
        safe_pip_install("python-docx>=1.1.0", import_name="docx")
    """
    modname = import_name or pkg_spec.split("==")[0].split("[")[0].replace("-", "_")
    try:
        importlib.import_module(modname)
        print(f"{modname}: already installed")
        return
    except Exception:
        pass

    print(f"Installing {pkg_spec} ...")
    cmd_quiet = [sys.executable, "-m", "pip", "install", "-q", pkg_spec]
    cmd_verbose = [sys.executable, "-m", "pip", "install", "--no-cache-dir", pkg_spec]

    try:
        subprocess.check_call(cmd_quiet)
        print(f"{pkg_spec}: OK")
    except subprocess.CalledProcessError:
        print(f"[Warning] Quiet install failed for {pkg_spec}. Retrying verbosely...")
        subprocess.check_call(cmd_verbose)
        print(f"{pkg_spec}: OK (after verbose retry)")

# --- Required packages (minimal, thesis-safe stack) ---
REQUIRED = [
    ("python-docx>=1.1.0", "docx"),
    ("docx2txt>=0.8", None),
    ("unidecode>=1.3.7", None),
    ("langdetect>=1.0.9", None),
    ("stanza==1.8.2", None),           # pinned for stable language models
    ("scikit-learn>=1.4", None),
    ("rake-nltk>=1.0.6", None),
    ("pyarrow>=14.0.2", None),
    ("matplotlib>=3.8", None),
]

for spec, imp in REQUIRED:
    safe_pip_install(spec, import_name=imp)

# --- Imports after installation ---
import numpy as np, pandas as pd
import json, re, hashlib, unicodedata, io, textwrap, warnings
from pathlib import Path
from datetime import datetime, timezone

# --- Environment report (for reproducibility appendix) ---
print("== Environment ==")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

for m in ["docx", "docx2txt", "stanza", "sklearn", "langdetect", "unidecode", "rake_nltk", "pyarrow", "matplotlib"]:
    try:
        importlib.import_module(m)
        print(f"{m}: OK")
    except Exception as e:
        print(f"{m}: ERROR -> {e}")

# --- Reproducibility seeds ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("\nEnvironment setup completed successfully.")


Installing python-docx>=1.1.0 ...
python-docx>=1.1.0: OK
Installing docx2txt>=0.8 ...
docx2txt>=0.8: OK
Installing unidecode>=1.3.7 ...
unidecode>=1.3.7: OK
Installing langdetect>=1.0.9 ...
langdetect>=1.0.9: OK
Installing stanza==1.8.2 ...
stanza==1.8.2: OK
Installing scikit-learn>=1.4 ...
scikit-learn>=1.4: OK
Installing rake-nltk>=1.0.6 ...
rake-nltk>=1.0.6: OK
Installing pyarrow>=14.0.2 ...
pyarrow>=14.0.2: OK
Installing matplotlib>=3.8 ...
matplotlib>=3.8: OK
== Environment ==
Python: 3.12.12
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
NumPy: 2.0.2
Pandas: 2.2.2
docx: OK
docx2txt: OK
stanza: OK
sklearn: OK
langdetect: OK
unidecode: OK
rake_nltk: OK
pyarrow: OK
matplotlib: OK

Environment setup completed successfully.


# Cell 2 — Project Configuration and Run Logger

In [ ]:
# ============================================================
# Cell 2 — Project Configuration and Run Logger
# ------------------------------------------------------------
# This cell defines project-wide constants such as data paths,
# language-country mappings, and year ranges. It also includes
# a simple JSONL-based logging utility for audit purposes.
# The logger records key parameters and events with timestamps.
# ============================================================

from typing import Dict, Any

# --- Country and language settings ---
COUNTRIES = {"EN": "United Kingdom", "DE": "Germany", "TR": "Turkey"}
YEARS = list(range(2019, 2024))  # Covers 2019–2023 inclusive
ARTICLES_PER_DOC_EXPECTED = 250  # Expected number of articles per DOCX file

# --- Directory structure (Google Drive-compatible) ---
# Adjust ROOT if using a different drive or mount point.
ROOT = Path("/content/Project_Dissertation")
DATA_DIR = ROOT / "data" / "unstructured" / "DATASET"
OUT_PROCESSED = ROOT / "data" / "processed"
ARTIFACTS = ROOT / "artifacts"

# Ensure that required folders exist
for p in [
    DATA_DIR,
    OUT_PROCESSED,
    ARTIFACTS / "models",
    ARTIFACTS / "figures",
    ARTIFACTS / "tables",
    ARTIFACTS / "logs",
]:
    p.mkdir(parents=True, exist_ok=True)

RUN_LOG = ARTIFACTS / "logs" / "pipeline_run.jsonl"

# --- Utility: Compute SHA-1 hash for data integrity checks ---
def sha1_of_file(path: Path) -> str:
    """Compute SHA-1 hash of a file for reproducibility tracking."""
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# --- Utility: Append structured events to JSONL log file ---
def append_log(event: Dict[str, Any]) -> None:
    """Append a dictionary-based event with timestamp to the run log."""
    event = dict(event)
    event["timestamp"] = datetime.now(timezone.utc).isoformat()
    with open(RUN_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(event, ensure_ascii=False) + "\n")

# --- Health check printout ---
print("== Configuration ==")
print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUT_PROCESSED:", OUT_PROCESSED)
print("ARTIFACTS:", ARTIFACTS)
print("RUN_LOG:", RUN_LOG)

# Initial configuration log entry
append_log({
    "event": "config_initialised",
    "countries": COUNTRIES,
    "years": YEARS,
    "expected_articles": ARTICLES_PER_DOC_EXPECTED
})

print("Configuration setup completed successfully.")


== Configuration ==
ROOT: /content/Project_Dissertation
DATA_DIR: /content/Project_Dissertation/data/unstructured/DATASET
OUT_PROCESSED: /content/Project_Dissertation/data/processed
ARTIFACTS: /content/Project_Dissertation/artifacts
RUN_LOG: /content/Project_Dissertation/artifacts/logs/pipeline_run.jsonl
Configuration setup completed successfully.


# Cell 3 — Mount Google Drive and Verify Dataset Structure

In [ ]:
# ============================================================
# Cell 3 — Mount Google Drive and Verify Dataset Structure
# ------------------------------------------------------------
# This cell mounts Google Drive (if applicable) and verifies
# that the dataset folders and .docx files required for the
# NLP analysis are available and accessible.
# ============================================================

from pathlib import Path
from google.colab import drive

# --- Mount Google Drive (no overwrite if already mounted) ---
drive.mount("/content/drive", force_remount=False)

# --- Define the dataset directory path ---
DATA_DIR = Path("/content/drive/MyDrive/Project Dissertation/AI_Refugee_Project/data/unstructured/DATASET")

# --- Verify directory existence ---
assert DATA_DIR.exists(), f"Error: DATA_DIR not found at {DATA_DIR}"

# --- Check expected language/country subfolders ---
subdirs = ["German", "Turkey", "United Kingdom"]
print("\n== Dataset Folder Check ==")
for sd in subdirs:
    path = DATA_DIR / sd
    status = "OK" if path.exists() else "MISSING"
    print(f"{sd:<18} → {status}  |  {path}")

# --- Count and preview available DOCX files ---
docx_files = list(DATA_DIR.rglob("*.docx"))
print(f"\nTotal .docx files found: {len(docx_files)}")
for name in sorted([p.name for p in docx_files])[:10]:
    print(" •", name)

# --- Optional: Record event in audit log (if logger defined) ---
if "append_log" in globals():
    append_log({
        "event": "drive_mounted",
        "data_dir": str(DATA_DIR),
        "docx_file_count": len(docx_files)
    })

print("\nData directory successfully verified.")


Mounted at /content/drive

== Dataset Folder Check ==
German             → OK  |  /content/drive/MyDrive/Project Dissertation/AI_Refugee_Project/data/unstructured/DATASET/German
Turkey             → OK  |  /content/drive/MyDrive/Project Dissertation/AI_Refugee_Project/data/unstructured/DATASET/Turkey
United Kingdom     → OK  |  /content/drive/MyDrive/Project Dissertation/AI_Refugee_Project/data/unstructured/DATASET/United Kingdom

Total .docx files found: 15
 • DE 2019.docx
 • DE 2020.docx
 • DE 2021.docx
 • DE 2022.docx
 • DE 2023.docx
 • EN 2019.docx
 • EN 2020.docx
 • EN 2021.docx
 • EN 2022.docx
 • EN 2023.docx

Data directory successfully verified.


# Cell 4 — File Discovery and Validation

In [ ]:
# ============================================================
# Cell 4 — File Discovery and Validation
# ------------------------------------------------------------
# This cell scans the dataset directory recursively to locate
# all .docx files. It validates naming patterns, ensures correct
# language–year combinations, checks approximate file size, and
# computes SHA-1 hashes for reproducibility.
# ============================================================

import re

# --- Expected matrix (3 languages × 5 years = 15 total files) ---
EXPECTED = [(lang, y) for lang in COUNTRIES.keys() for y in YEARS]

# --- Collect .docx candidates recursively ---
all_docx = list(DATA_DIR.rglob("*.docx"))
print(f"Scanning directory: {DATA_DIR}")
print(f"Total .docx files detected (recursive): {len(all_docx)}")

# --- Naming pattern: e.g., "EN 2019.docx", "DE_2020.docx", "TR-2021.docx" ---
name_pattern = re.compile(r"^(EN|DE|TR)[\s_\-]?(\d{4})\.docx$", flags=re.IGNORECASE)

records, unmatched_files = [], []

# --- File validation loop ---
for path in all_docx:
    filename = path.name
    match = name_pattern.match(filename)
    if not match:
        unmatched_files.append(filename)
        continue

    lang = match.group(1).upper()
    year = int(match.group(2))
    if lang not in COUNTRIES or year not in YEARS:
        unmatched_files.append(filename)
        continue

    country = COUNTRIES[lang]
    try:
        size_ok = path.stat().st_size > 10_000  # sanity check (~10 KB minimum)
    except Exception:
        size_ok = False

    try:
        sha1 = sha1_of_file(path)
    except Exception:
        sha1 = "NA"

    records.append({
        "file": filename,
        "path": str(path),
        "size_bytes": path.stat().st_size if path.exists() else 0,
        "lang": lang,
        "year": year,
        "country": country,
        "size_ok": size_ok,
        "sha1": sha1
    })

# --- Convert to DataFrame ---
df_files = pd.DataFrame(records)

# --- Compare expected vs. found files ---
found_pairs = set((r["lang"], r["year"]) for r in records)
expected_pairs = set(EXPECTED)
missing_pairs = sorted(expected_pairs - found_pairs)
extra_count = len(unmatched_files)

print(f"\nExpected files: {len(EXPECTED)} | Valid matches found: {len(records)}")
if missing_pairs:
    print("Missing (language, year) combinations:", missing_pairs)
if extra_count:
    print(f"Files not matching naming pattern or year range: {extra_count}")

# --- Display summary table if data available ---
if not df_files.empty:
    df_files = df_files.sort_values(["lang", "year", "file"]).reset_index(drop=True)
    display(df_files)
else:
    print("No valid files found. Please verify your DATA_DIR path and file names "
          "(expected format: 'EN 2019.docx', 'DE_2020.docx', 'TR-2021.docx').")

# --- Log event (if logger available) ---
if "append_log" in globals():
    append_log({
        "event": "file_discovery_complete",
        "valid_files": int(len(df_files)),
        "missing_pairs": missing_pairs,
        "unmatched_files_count": extra_count,
        "data_dir": str(DATA_DIR)
    })

print("\nFile discovery and validation completed successfully.")


Scanning directory: /content/drive/MyDrive/Project Dissertation/AI_Refugee_Project/data/unstructured/DATASET
Total .docx files detected (recursive): 15

Expected files: 15 | Valid matches found: 15


,file,path,size_bytes,lang,year,country,size_ok,sha1
0,DE 2019.docx,/content/drive/MyDrive/Project Dissertation/AI...,163261,DE,2019,Germany,True,c9b5a7b9ea73cd0e0aad40187904cb8e555e2f6b
1,DE 2020.docx,/content/drive/MyDrive/Project Dissertation/AI...,186890,DE,2020,Germany,True,5f3b45aaf6395e4f3e62e0e1602d9e9c34ed7b59
2,DE 2021.docx,/content/drive/MyDrive/Project Dissertation/AI...,157762,DE,2021,Germany,True,c29cabd4838615aa8c4f0b4317c30e3aff4b9e95
3,DE 2022.docx,/content/drive/MyDrive/Project Dissertation/AI...,168553,DE,2022,Germany,True,fb9ef2bf1b3a1959f67fe5fd987af889bc6d1890
4,DE 2023.docx,/content/drive/MyDrive/Project Dissertation/AI...,188414,DE,2023,Germany,True,a7b71ee4c318ca2e48b1cc53c67b61a06234c56b
5,EN 2019.docx,/content/drive/MyDrive/Project Dissertation/AI...,299437,EN,2019,United Kingdom,True,e21fe1ef0c7f9b70af76baefe181da7413085640
6,EN 2020.docx,/content/drive/MyDrive/Project Dissertation/AI...,324466,EN,2020,United Kingdom,True,f9101df9e77d1a5fc6e0af076f6942bf528c5797
7,EN 2021.docx,/content/drive/MyDrive/Project Dissertation/AI...,314128,EN,2021,United Kingdom,True,72d87cbbd72e5415748230c5b371a0a49271457e
8,EN 2022.docx,/content/drive/MyDrive/Project Dissertation/AI...,318492,EN,2022,United Kingdom,True,e4dbfb37a65aa01a5453ea5a6e388d9bc86d9a55
9,EN 2023.docx,/content/drive/MyDrive/Project Dissertation/AI...,312286,EN,2023,United Kingdom,True,0710aaa108a3c1c396a375709f677bc6833a3601



File discovery and validation completed successfully.


# Cell 5 — Ingest DOCX Files into Paragraph-Level Raw Table

In [ ]:
# ============================================================
# Cell 5 — Ingest DOCX Files into Paragraph-Level Raw Table
# ------------------------------------------------------------
# This cell reads all DOCX news source files, extracts paragraph-
# level text, and builds a raw dataframe (df_raw) suitable for
# further NLP processing. It also generates basic diagnostics
# such as paragraph counts, character counts, and naive sentence
# estimates to check data completeness.
# ============================================================

from pathlib import Path
from docx import Document as DocxDocument
import docx2txt, re, json
import pandas as pd
from unidecode import unidecode

# --- Regular expression for filename parsing ---
NAME_RE = re.compile(r"^(EN|DE|TR)[\s_\-]?(\d{4})\.docx$", flags=re.IGNORECASE)

# --- Helper: read paragraphs from a DOCX file (fallback to docx2txt if needed) ---
def read_docx_paragraphs(path: Path):
    """Extract paragraphs from a Word document. Uses python-docx primarily,
    but falls back to docx2txt for files with too few non-empty lines."""
    try:
        doc = DocxDocument(str(path))
        paragraphs = [p.text.strip("\n\r ") for p in doc.paragraphs]
        if sum(1 for t in paragraphs if t) < 20:
            raise RuntimeError("fallback_to_docx2txt")
        return paragraphs
    except Exception:
        text = docx2txt.process(str(path)) or ""
        return [line.strip() for line in text.splitlines()]

# --- Helper: extract language and year from filename ---
def file_lang_year(path: Path):
    """Extract language code and year from filename, e.g. 'EN 2019.docx'."""
    m = NAME_RE.match(path.name)
    if not m:
        return None, None
    lang = m.group(1).upper()
    year = int(m.group(2))
    return lang, year

# --- Discover all valid DOCX files (recursive under DATA_DIR) ---
docx_paths = [p for p in DATA_DIR.rglob("*.docx") if NAME_RE.match(p.name)]
docx_paths = sorted(docx_paths, key=lambda p: (p.name[:2], p.name))

# --- Diagnostic and data collection lists ---
diag_rows, raw_rows = [], []

for path in docx_paths:
    lang, year = file_lang_year(path)
    if lang is None or year not in YEARS or lang not in COUNTRIES:
        continue

    country = COUNTRIES[lang]
    doc_id = f"{lang}_{year}"

    paragraphs = read_docx_paragraphs(path)
    n_paras = len(paragraphs)
    n_nonempty = sum(1 for t in paragraphs if t and t.strip())
    n_chars = sum(len(t) for t in paragraphs if t)
    naive_sentences = sum(len(re.split(r"[.!?]+", t)) for t in paragraphs if t)

    diag_rows.append({
        "file": path.name,
        "path": str(path),
        "doc_id": doc_id,
        "lang": lang,
        "year": year,
        "country": country,
        "n_paragraphs": n_paras,
        "n_nonempty": n_nonempty,
        "n_chars": n_chars,
        "naive_sentence_count": naive_sentences
    })

    for i, text in enumerate(paragraphs):
        raw_rows.append({
            "doc_id": doc_id,
            "country": country,
            "lang": lang,
            "year": year,
            "para_idx": i,
            "text_raw": text if text is not None else ""
        })

# --- Create dataframes ---
df_diag = pd.DataFrame(diag_rows).sort_values(["lang", "year", "file"])
df_raw = pd.DataFrame(raw_rows)

print(f"Valid source files: {len(df_diag)}")
print(f"Paragraph-level rows collected: {len(df_raw)}")

# Display brief diagnostics
display(df_diag.head(10))
display(df_raw.head(10))

# --- Save raw dataframe to disk ---
OUT_PROCESSED.mkdir(parents=True, exist_ok=True)
raw_path = OUT_PROCESSED / "df_raw.parquet"
df_raw.to_parquet(raw_path, index=False)

# --- Log event ---
if "append_log" in globals():
    append_log({
        "event": "ingestion_complete",
        "file_count": int(len(df_diag)),
        "paragraph_rows": int(len(df_raw)),
        "output_path": str(raw_path)
    })

print("\nParagraph-level ingestion completed successfully.")


Valid source files: 15
Paragraph-level rows collected: 24569


,file,path,doc_id,lang,year,country,n_paragraphs,n_nonempty,n_chars,naive_sentence_count
0,DE 2019.docx,/content/drive/MyDrive/Project Dissertation/AI...,DE_2019,DE,2019,Germany,1529,1276,315439,5347
1,DE 2020.docx,/content/drive/MyDrive/Project Dissertation/AI...,DE_2020,DE,2020,Germany,1578,1325,334833,5650
2,DE 2021.docx,/content/drive/MyDrive/Project Dissertation/AI...,DE_2021,DE,2021,Germany,1426,1173,274600,4843
3,DE 2022.docx,/content/drive/MyDrive/Project Dissertation/AI...,DE_2022,DE,2022,Germany,1451,1198,276572,4970
4,DE 2023.docx,/content/drive/MyDrive/Project Dissertation/AI...,DE_2023,DE,2023,Germany,1611,1358,349375,6043
5,EN 2019.docx,/content/drive/MyDrive/Project Dissertation/AI...,EN_2019,EN,2019,United Kingdom,1954,1701,521492,7710
6,EN 2020.docx,/content/drive/MyDrive/Project Dissertation/AI...,EN_2020,EN,2020,United Kingdom,1971,1718,542950,7559
7,EN 2021.docx,/content/drive/MyDrive/Project Dissertation/AI...,EN_2021,EN,2021,United Kingdom,1964,1711,542789,7788
8,EN 2022.docx,/content/drive/MyDrive/Project Dissertation/AI...,EN_2022,EN,2022,United Kingdom,1986,1733,544029,7830
9,EN 2023.docx,/content/drive/MyDrive/Project Dissertation/AI...,EN_2023,EN,2023,United Kingdom,1971,1718,542969,7597


,doc_id,country,lang,year,para_idx,text_raw
0,DE_2019,Germany,DE,2019,0,
1,DE_2019,Germany,DE,2019,1,
2,DE_2019,Germany,DE,2019,2,
3,DE_2019,Germany,DE,2019,3,Results for: Flüchtling OR Asylbewerber OR Mig...
4,DE_2019,Germany,DE,2019,4,News
5,DE_2019,Germany,DE,2019,5,
6,DE_2019,Germany,DE,2019,6,Flüchtling darf zurück Deutschland muss einen ...
7,DE_2019,Germany,DE,2019,7,"Nürnberger Nachrichten | Aug 15, 2019 | POLITI..."
8,DE_2019,Germany,DE,2019,8,... Flüchtling darf zurück Deutschland muss ei...
9,DE_2019,Germany,DE,2019,9,... München.Rund ein Jahr nach Abschluss des R...



Paragraph-level ingestion completed successfully.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cell 6 — Article Segmentation and Count Validation

In [ ]:
# ============================================================
# Cell 6 — Article Segmentation and Count Validation
# ------------------------------------------------------------
# This cell identifies how many numbered news articles exist
# within each DOCX file. It uses two complementary approaches:
#   1. Word-native numbering: detects top-level list items
#      (ilvl = 0) using python-docx numbering metadata.
#   2. Plain-text numeric anchors: detects leading numbers such
#      as "1.", "2)", or "3 - " at the start of lines.
# The resulting counts are summarised per file and exported
# for reproducibility and appendix reference.
# ============================================================

import re, os, unicodedata, json, hashlib, warnings
from pathlib import Path
import pandas as pd
from docx import Document as DocxDocument

# --- Directory preparation ---
ROOT_OUT = Path("/content/Project_Dissertation")
TABLES_DIR = ROOT_OUT / "artifacts" / "tables"
PROCESSED_DIR = ROOT_OUT / "data" / "processed"
for d in [TABLES_DIR, PROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Define language-folder mapping ---
FOLDER_LANG = {
    "German": ("DE", "Germany"),
    "Turkey": ("TR", "Turkey"),
    "United Kingdom": ("EN", "United Kingdom"),
}

# --- Utility: compute SHA-1 hash for data verification ---
def sha1_of_file(path: Path) -> str:
    """Compute SHA-1 hash of a file for reproducibility tracking."""
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# --- Utility: normalize text (standardise dashes, line breaks) ---
def normalize_text_for_scan(txt: str) -> str:
    txt = txt.replace("\r\n", "\n").replace("\r", "\n")
    txt = unicodedata.normalize("NFC", txt)
    txt = txt.replace("–", "-").replace("—", "-")
    return txt

# --- Helper: read Word paragraphs safely ---
def read_docx_paragraphs(path: Path):
    """Return list of paragraph objects from a DOCX file."""
    doc = DocxDocument(str(path))
    return doc.paragraphs

# --- Method A: count based on Word-native numbering (preferred) ---
def count_by_numbering(path: Path) -> tuple[int, dict]:
    """
    Detect top-level numbered paragraphs (ilvl = 0) grouped by numId.
    Returns (count, details) for the numbering sequence with the
    largest group size.
    """
    paras = read_docx_paragraphs(path)
    groups = {}
    for i, p in enumerate(paras):
        pPr = p._p.pPr
        if pPr is None or pPr.numPr is None:
            continue
        ilvl = pPr.numPr.ilvl
        numId = pPr.numPr.numId
        try:
            ilvl_val = int(ilvl.val) if ilvl is not None else None
            numId_val = int(numId.val) if numId is not None else None
        except Exception:
            ilvl_val, numId_val = None, None
        if ilvl_val == 0 and numId_val is not None:
            groups.setdefault(numId_val, []).append(i)

    if not groups:
        return 0, {"method": "numbering", "numId": None, "detail": "no ilvl==0 found"}

    numId_best, idx_list = max(groups.items(), key=lambda kv: len(kv[1]))
    return len(idx_list), {
        "method": "numbering",
        "numId": numId_best,
        "detail": f"ilvl=0 count={len(idx_list)}"
    }

# --- Method B: count using plain-text numeric anchors (fallback) ---
ANCHOR_INLINE = re.compile(r"^\s{0,3}(\d{1,3})\s*([.)-])(?:\s+|$)")
ANCHOR_BARE = re.compile(r"^\s{0,3}(\d{1,3})\s*$")

def count_by_text(path: Path) -> tuple[int, dict]:
    """
    Identify numbered lines based on plain-text patterns like
    '1.', '2)', or '3 -'. Builds a contiguous 1..N sequence
    from first occurrences and returns N.
    """
    doc = DocxDocument(str(path))
    lines = [normalize_text_for_scan((p.text or "")) for p in doc.paragraphs]

    candidates = []
    for i, line in enumerate(lines):
        m = ANCHOR_INLINE.match(line) or ANCHOR_BARE.match(line)
        if m:
            n = int(m.group(1))
            if 1 <= n <= 999:
                candidates.append((i, n))

    if not candidates:
        return 0, {"method": "text", "detail": "no numeric anchors found"}

    first_idx_for_num = {}
    for i, n in candidates:
        if n not in first_idx_for_num:
            first_idx_for_num[n] = i

    N, expected = 0, 1
    while expected in first_idx_for_num:
        N += 1
        expected += 1

    return N, {"method": "text", "detail": f"contiguous 1..{N} anchors"}

# --- Count news items per file ---
rows = []

for folder, (lang, country) in FOLDER_LANG.items():
    base = DATA_DIR / folder
    if not base.exists():
        warnings.warn(f"Folder not found: {base}")
        continue

    files = sorted([p for p in base.glob("*.docx") if any(str(y) in p.name for y in YEARS)])
    for path in files:
        m = re.search(r"(20\d{2})", path.name)
        if not m:
            continue
        year = int(m.group(1))

        # Try numbering-based detection first, then fallback to text-based
        cnt_num, info_num = count_by_numbering(path)
        if cnt_num >= 1:
            count, method, detail = cnt_num, info_num["method"], info_num["detail"]
        else:
            cnt_txt, info_txt = count_by_text(path)
            count, method, detail = cnt_txt, info_txt["method"], info_txt["detail"]

        rows.append({
            "doc_id": f"{lang}_{year}",
            "lang": lang,
            "year": year,
            "country": country,
            "found_articles": count,
            "method": method,
            "detail": detail,
            "file": path.name,
            "sha1": sha1_of_file(path)
        })

# --- Create and display summary ---
df_counts = pd.DataFrame(rows).sort_values(["lang", "year"]).reset_index(drop=True)
print("== News Item Counts per Document ==")
display(df_counts[["doc_id", "lang", "year", "country", "found_articles", "method"]])

# --- Save results for reproducibility (appendix use) ---
out_csv = TABLES_DIR / "segmentation_counts.csv"
df_counts.to_csv(out_csv, index=False, encoding="utf-8")
print("\nSaved segmentation summary to:", out_csv)

# --- Optional audit log entry ---
if "append_log" in globals():
    append_log({
        "event": "segmentation_completed",
        "files_processed": int(len(df_counts)),
        "output_csv": str(out_csv)
    })

print("\nArticle segmentation completed successfully.")


== News Item Counts per Document ==


,doc_id,lang,year,country,found_articles,method
0,DE_2019,DE,2019,Germany,250,numbering
1,DE_2020,DE,2020,Germany,250,numbering
2,DE_2021,DE,2021,Germany,250,numbering
3,DE_2022,DE,2022,Germany,250,numbering
4,DE_2023,DE,2023,Germany,250,numbering
5,EN_2019,EN,2019,United Kingdom,250,numbering
6,EN_2020,EN,2020,United Kingdom,250,numbering
7,EN_2021,EN,2021,United Kingdom,250,numbering
8,EN_2022,EN,2022,United Kingdom,250,numbering
9,EN_2023,EN,2023,United Kingdom,250,numbering



Saved segmentation summary to: /content/Project_Dissertation/artifacts/tables/segmentation_counts.csv

Article segmentation completed successfully.


# Cell 7 — Cleaning, Normalisation, and Dataset Preparation

In [ ]:
# ============================================================
# Cell 7 — Cleaning, Normalisation, and Dataset Preparation
# ------------------------------------------------------------
# This cell reconstructs the df_news table (if not already in memory)
# and applies consistent text normalisation for multilingual data.
# Key steps:
#   1. Segment each DOCX file into 250 news articles using Word-native numbering.
#   2. Produce cleaned text columns suitable for NLP analysis.
#   3. Generate diagnostic summaries (token statistics, top tokens, spot samples).
#   4. Save the cleaned dataset (df_news_clean) to Parquet format.
# ============================================================

import re
import math
import pandas as pd
import numpy as np
from pathlib import Path
from docx import Document as DocxDocument
from unidecode import unidecode
import matplotlib.pyplot as plt
import unicodedata
import random

# ---------------------------
# 0) Paths and Utilities
# ---------------------------
ROOT_OUT = Path("/content/Project_Dissertation")
DATA_DIR = Path("/content/drive/MyDrive/Project Dissertation/AI_Refugee_Project/data/unstructured/DATASET")
ARTIFACTS = ROOT_OUT / "artifacts"
FIG_DIR   = ARTIFACTS / "figures"
TAB_DIR   = ARTIFACTS / "tables"
PROC_DIR  = ROOT_OUT / "data" / "processed"
for p in [FIG_DIR, TAB_DIR, PROC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

FOLDER_LANG = {
    "German": ("DE", "Germany"),
    "Turkey": ("TR", "Turkey"),
    "United Kingdom": ("EN", "United Kingdom")
}
YEARS = list(range(2019, 2024))

# ---------------------------
# 1) Build df_news (if missing)
# ---------------------------
def is_numbered_top_level(p):
    """Return True if paragraph is a top-level numbered item (ilvl == 0)."""
    pPr = p._p.pPr
    if pPr is None or pPr.numPr is None:
        return False
    ilvl = pPr.numPr.ilvl
    try:
        ilvl_val = int(ilvl.val)
    except Exception:
        ilvl_val = None
    return ilvl_val == 0

def segment_doc_by_numbering(doc_path: Path, lang: str, year: int, country: str) -> pd.DataFrame:
    """
    Segment DOCX content into individual news articles based on
    top-level numbered anchors. The first paragraph of each segment
    is treated as the title, and subsequent paragraphs form the body.
    """
    doc = DocxDocument(str(doc_path))
    paras = [(i, (p.text or "").strip(), p) for i, p in enumerate(doc.paragraphs)]
    anchors = [i for i, txt, p in paras if is_numbered_top_level(p)]
    if not anchors:
        raise RuntimeError(f"No top-level numbered anchors found in {doc_path.name}.")

    spans, rows = [], []
    for k, a in enumerate(anchors):
        b = anchors[k + 1] if k + 1 < len(anchors) else len(paras)
        spans.append((k + 1, a, b))

    for art_idx, a, b in spans:
        title = paras[a][1]
        body_lines = [paras[j][1] for j in range(a, b)]
        body = "\n".join(body_lines).strip()
        n_chars = len(body)
        n_tokens_est = len(re.findall(r"\w+", body, flags=re.UNICODE))
        rows.append({
            "doc_id": f"{lang}_{year}",
            "lang": lang,
            "year": int(year),
            "country": country,
            "art_idx": art_idx,
            "title_raw": title,
            "body_raw": body,
            "n_chars": n_chars,
            "n_tokens_est": n_tokens_est
        })
    return pd.DataFrame(rows)

def build_df_news_from_docs(base_dir: Path) -> pd.DataFrame:
    """Assemble df_news from all DOCX files (German, Turkish, English)."""
    all_parts = []
    for folder, (lang, country) in FOLDER_LANG.items():
        fpath = base_dir / folder
        for year in YEARS:
            docx = fpath / f"{lang} {year}.docx"
            if not docx.exists():
                continue
            part = segment_doc_by_numbering(docx, lang=lang, year=year, country=country)
            all_parts.append(part)
    if not all_parts:
        raise RuntimeError("No DOCX documents found for segmentation.")
    return pd.concat(all_parts, axis=0, ignore_index=True)

# Build df_news only if not already loaded
if "df_news" not in globals():
    df_news = build_df_news_from_docs(DATA_DIR)
    raw_out = PROC_DIR / "df_news_raw.parquet"
    df_news.to_parquet(raw_out, index=False)
    print(f"df_news constructed from DOCX files and saved to: {raw_out}")
else:
    print("Using existing df_news object in memory.")

# ---------------------------
# 2) Text Cleaning / Normalisation
# ---------------------------
URL_RE   = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w{2,}\b", flags=re.IGNORECASE)
NUM_RE   = re.compile(r"\b\d+([.,]\d+)?\b", flags=re.UNICODE)

def normalize_quotes_dashes(s: str) -> str:
    """Standardise quotation marks and dashes."""
    s = s.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    s = s.replace("–", "-").replace("—", "-")
    return unicodedata.normalize("NFC", s)

def strip_boilerplate(s: str) -> str:
    """Remove footer or legal phrases found across sources."""
    return re.sub(
        r"(All rights reserved|Alle Rechte vorbehalten|Tüm hakları saklıdır).*$",
        " ", s, flags=re.IGNORECASE | re.MULTILINE
    )

def clean_text(s: str) -> str:
    """Light-touch cleaning: remove URLs, emails, numbers, and extra spaces."""
    if not s:
        return ""
    s = normalize_quotes_dashes(strip_boilerplate(s))
    s = URL_RE.sub(" ", s)
    s = EMAIL_RE.sub(" ", s)
    s = NUM_RE.sub(" <NUM> ", s)
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\s*\n\s*\n\s*", "\n\n", s)
    return s.strip()

# Apply cleaning and ASCII mirrors
df_news["title_clean"] = df_news["title_raw"].fillna("").map(clean_text)
df_news["body_clean"]  = df_news["body_raw"].fillna("").map(clean_text)
df_news["title_clean_ascii"] = df_news["title_clean"].map(unidecode)
df_news["body_clean_ascii"]  = df_news["body_clean"].map(unidecode)

# Add token/character length columns
df_news["len_tokens"] = df_news["body_clean"].str.split().map(len)
df_news["len_chars"]  = df_news["body_clean"].str.len()

# ---------------------------
# 3) Diagnostics and Artifacts
# ---------------------------
# (a) Token length distributions — three subplots side-by-side (shared bins)
ORDER = ["Germany", "United Kingdom", "Turkey"]
LANG_BY_COUNTRY = {"Germany":"DE", "United Kingdom":"EN", "Turkey":"TR"}

# Build a common bins array so histograms are directly comparable
all_lengths = df_news["len_tokens"].values
min_len, max_len = int(np.min(all_lengths)), int(np.max(all_lengths))
bins = np.linspace(min_len, max_len, 40)

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, country in zip(axes, ORDER):
    lang = LANG_BY_COUNTRY[country]
    subset = df_news[df_news["lang"] == lang]
    ax.hist(subset["len_tokens"], bins=bins)
    ax.set_title(f"Token length distribution — {country}")
    ax.set_xlabel("Tokens per article")
    ax.set_ylabel("Number of articles")
    ax.grid(alpha=0.25, axis="y")
fig.tight_layout()
out = FIG_DIR / "token_len_dists_by_country.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)

# (b) Top-20 frequent tokens (rough frequency count)
def top_tokens(series: pd.Series, topk=20):
    counts = {}
    for text in series.dropna().tolist():
        for tok in text.split():
            tok = tok.strip()
            if tok:
                counts[tok] = counts.get(tok, 0) + 1
    top_items = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:topk]
    return pd.DataFrame(top_items, columns=["token", "freq"])

tops_all = []
for lang in sorted(df_news["lang"].unique()):
    tdf = top_tokens(df_news.loc[df_news["lang"] == lang, "body_clean"], topk=20)
    tdf.insert(0, "lang", lang)
    tops_all.append(tdf)
df_top_tokens = pd.concat(tops_all, ignore_index=True)
df_top_tokens.to_csv(TAB_DIR / "top20_tokens_by_language.csv", index=False, encoding="utf-8")

# (c) Spot checks: random 3 samples per language-year
spot_rows = []
random.seed(42)
for (lang, year), grp in df_news.groupby(["lang", "year"]):
    sample = grp.sample(n=min(3, len(grp)), random_state=42)
    for _, r in sample.iterrows():
        spot_rows.append({
            "doc_id": r["doc_id"],
            "lang": lang,
            "year": int(year),
            "art_idx": int(r["art_idx"]),
            "title_clean": r["title_clean"][:160],
            "body_head": (r["body_clean"][:200] + ("…" if len(r["body_clean"]) > 200 else "")),
            "len_tokens": int(r["len_tokens"])
        })
df_spots = pd.DataFrame(spot_rows).sort_values(["lang", "year", "art_idx"])
df_spots.to_csv(TAB_DIR / "spotchecks_samples.csv", index=False, encoding="utf-8")

# (d) Summary statistics per language
summary = (
    df_news.groupby("lang")
    .agg(
        articles=("art_idx", "size"),
        avg_tokens=("len_tokens", "mean"),
        median_tokens=("len_tokens", "median"),
        avg_chars=("len_chars", "mean")
    )
    .reset_index()
)
summary.to_csv(TAB_DIR / "cleaning_summary_by_language.csv", index=False, encoding="utf-8")

# ---------------------------
# 4) Save cleaned dataset
# ---------------------------
clean_path = PROC_DIR / "df_news_clean.parquet"
df_news.to_parquet(clean_path, index=False)
print("Cleaned dataset saved to:", clean_path)

# --- Optional audit log ---
if "append_log" in globals():
    append_log({
        "event": "cleaning_completed",
        "rows": int(df_news.shape[0]),
        "languages": sorted(df_news["lang"].unique().tolist()),
        "output_parquet": str(clean_path),
        "top_tokens_csv": str(TAB_DIR / "top20_tokens_by_language.csv"),
        "spotchecks_csv": str(TAB_DIR / "spotchecks_samples.csv"),
        "summary_csv": str(TAB_DIR / "cleaning_summary_by_language.csv")
    })

print("\nCleaning and normalisation completed successfully.")


df_news constructed from DOCX files and saved to: /content/Project_Dissertation/data/processed/df_news_raw.parquet
Cleaned dataset saved to: /content/Project_Dissertation/data/processed/df_news_clean.parquet

Cleaning and normalisation completed successfully.


# Cell 8 — Language Detection and Consistency Checks

In [ ]:
# ============================================================
# Cell 8 — Language Detection and Consistency Checks
# ------------------------------------------------------------
# This cell verifies that each reconstructed article is written
# in the expected language (EN, DE, or TR). It uses the
# langdetect library to identify the dominant language and
# compares it with the file’s expected language label.
#
# Outputs:
#   • df_langdetect.parquet  – per-article detection results
#   • language_mismatches.csv – individual mismatch rows
#   • language_mismatch_summary.csv – mismatch counts by document
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from langdetect import detect_langs, LangDetectException, DetectorFactory

# --- Ensure deterministic results ---
DetectorFactory.seed = 42

ROOT_OUT = Path("/content/Project_Dissertation")
PROC_DIR = ROOT_OUT / "data" / "processed"
TAB_DIR = ROOT_OUT / "artifacts" / "tables"
PROC_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

# --- Load dataset ---
if "df_news" not in globals():
    df_news = pd.read_parquet(PROC_DIR / "df_news_clean.parquet")

# --- Prepare input for language detection ---
def assemble_text(row, max_chars=1500):
    """Concatenate title and body, truncated for efficiency."""
    text = f"{(row.get('title_clean') or '').strip()}\n{(row.get('body_clean') or '').strip()}"
    return text[:max_chars].strip()

df_in = df_news.copy()
df_in["text_for_langdetect"] = df_in.apply(assemble_text, axis=1)

EXPECTED_ISO = {"EN": "en", "DE": "de", "TR": "tr"}

# --- Language detection helper ---
def detect_top_lang(text: str):
    """
    Return (language_code, probability). On failure or very short text,
    returns ('unk', 0.0). Only the top hypothesis from detect_langs is used.
    """
    text = (text or "").strip()
    if len(text) < 20:
        return "unk", 0.0
    try:
        hyps = detect_langs(text)
        if not hyps:
            return "unk", 0.0
        top = max(hyps, key=lambda x: x.prob)
        return top.lang, float(top.prob)
    except LangDetectException:
        return "unk", 0.0
    except Exception:
        return "unk", 0.0

# --- Run detection across all rows ---
langs, probs = [], []
for text in df_in["text_for_langdetect"]:
    l, p = detect_top_lang(text)
    langs.append(l)
    probs.append(p)

df_in["detected_lang"] = langs
df_in["detected_prob"] = probs

# --- Evaluate consistency ---
df_in["expected_lang_iso"] = df_in["lang"].map(EXPECTED_ISO)
df_in["manual_review"] = (
    (df_in["detected_lang"] != df_in["expected_lang_iso"]) &
    (df_in["detected_lang"] != "unk")
)

# --- Summaries ---
doc_summary = (
    df_in.groupby(["doc_id", "lang", "year", "country"], as_index=False)
         .agg(total=("art_idx", "size"),
              mismatches=("manual_review", "sum"))
)
doc_summary["mismatch_pct"] = (
    doc_summary["mismatches"] / doc_summary["total"] * 100
).round(2)

print("== Per-Document Mismatch Summary ==")
display(doc_summary.sort_values(["lang", "year"]))

print("\n== Expected vs Detected Languages ==")
conf = pd.crosstab(df_in["expected_lang_iso"], df_in["detected_lang"]).sort_index()
display(conf)

# --- Preview of mismatch cases ---
mismatches = df_in[df_in["manual_review"]].copy()
offenders_preview = mismatches.head(10)[
    ["doc_id", "art_idx", "expected_lang_iso", "detected_lang", "detected_prob", "title_clean"]
]
if not offenders_preview.empty:
    print("\n== Example Mismatch Cases (preview) ==")
    display(offenders_preview)
else:
    print("\nNo language mismatches detected.")

# --- Save artifacts ---
langdetect_out = PROC_DIR / "df_langdetect.parquet"
mismatch_csv = TAB_DIR / "language_mismatches.csv"
summary_csv = TAB_DIR / "language_mismatch_summary.csv"

df_in.to_parquet(langdetect_out, index=False)
mismatches.to_csv(mismatch_csv, index=False, encoding="utf-8")
doc_summary.to_csv(summary_csv, index=False, encoding="utf-8")

print("\nSaved:")
print(f" - Detection results: {langdetect_out}")
print(f" - Mismatch rows:     {mismatch_csv}")
print(f" - Summary table:     {summary_csv}")

# --- Optional run log ---
if "append_log" in globals():
    append_log({
        "event": "language_detection_completed",
        "rows": int(df_in.shape[0]),
        "mismatches": int(mismatches.shape[0]),
        "df_langdetect": str(langdetect_out),
        "mismatch_csv": str(mismatch_csv),
        "summary_csv": str(summary_csv)
    })

print("\nLanguage detection and consistency checks completed successfully.")


== Per-Document Mismatch Summary ==


,doc_id,lang,year,country,total,mismatches,mismatch_pct
0,DE_2019,DE,2019,Germany,250,0,0.0
1,DE_2020,DE,2020,Germany,250,0,0.0
2,DE_2021,DE,2021,Germany,250,0,0.0
3,DE_2022,DE,2022,Germany,250,0,0.0
4,DE_2023,DE,2023,Germany,250,0,0.0
5,EN_2019,EN,2019,United Kingdom,250,0,0.0
6,EN_2020,EN,2020,United Kingdom,250,0,0.0
7,EN_2021,EN,2021,United Kingdom,250,0,0.0
8,EN_2022,EN,2022,United Kingdom,250,0,0.0
9,EN_2023,EN,2023,United Kingdom,250,0,0.0



== Expected vs Detected Languages ==


detected_lang,de,en,tr
expected_lang_iso,,,
de,1250,0,0
en,0,1250,0
tr,0,0,1250



No language mismatches detected.

Saved:
 - Detection results: /content/Project_Dissertation/data/processed/df_langdetect.parquet
 - Mismatch rows:     /content/Project_Dissertation/artifacts/tables/language_mismatches.csv
 - Summary table:     /content/Project_Dissertation/artifacts/tables/language_mismatch_summary.csv

Language detection and consistency checks completed successfully.


# Cell 9a — Feature Extraction (TF-IDF + RAKE)

In [ ]:
# ============================================================
# Cell 9a — Feature Extraction (TF-IDF + RAKE)
# ------------------------------------------------------------
# Build language-specific TF-IDF vectorizers (EN/DE/TR) on
# title + body text (cleaned). Extract top TF-IDF terms and
# RAKE keyphrases for each article. Aggregate TF-IDF terms
# by (language × year) and (country × year).
# ============================================================

import re, json, pickle
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

ROOT_OUT = Path("/content/Project_Dissertation")
PROC_DIR  = ROOT_OUT / "data" / "processed"
TAB_DIR   = ROOT_OUT / "artifacts" / "tables"
MODEL_DIR = ROOT_OUT / "artifacts" / "models"
for p in [PROC_DIR, TAB_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if "df_news" not in globals():
    df_news = pd.read_parquet(PROC_DIR / "df_news_clean.parquet")

LANGS = ["EN", "DE", "TR"]
MIN_DF = 5
NGRAMS = (1, 2)
TOPN_TERMS_PER_ARTICLE = 10
TOPN_TERMS_AGGR = 25

def build_tfidf_for_language(df_lang: pd.DataFrame):
    docs = (df_lang["title_clean"].fillna("") + "\n" + df_lang["body_clean"].fillna("")).tolist()
    vec = TfidfVectorizer(
        analyzer="word",
        ngram_range=NGRAMS,
        min_df=MIN_DF,
        token_pattern=r"(?u)\b\w+\b",
        lowercase=False
    )
    X = vec.fit_transform(docs)
    feats = np.array(vec.get_feature_names_out())
    return vec, X, docs, feats

def top_terms_from_row(X_row, feature_names, topn=TOPN_TERMS_PER_ARTICLE):
    coo = X_row.tocoo()
    if coo.nnz == 0:
        return []
    order = np.argsort(coo.data)[::-1][:topn]
    terms = [f"{feature_names[j]}:{coo.data[k]:.4f}" for k, j in zip(order, coo.col[order])]
    return terms

STOP_EN = {
    "the","a","an","and","or","but","if","while","with","without","of","to","in","on","for","by","at","from",
    "is","are","was","were","be","been","being","this","that","these","those","it","its","as","about","into",
    "over","under","between","within","after","before","not","no","so","such","than","then","too","very"
}
STOP_DE = {
    "der","die","das","ein","eine","und","oder","aber","wenn","während","mit","ohne","von","zu","im","in","auf",
    "für","bei","an","aus","ist","sind","war","waren","sein","dies","das","diese","jene","es","als","über",
    "unter","zwischen","innerhalb","nach","vor","nicht","kein","so","solche","dann","sehr"
}
STOP_TR = {
    "ve","veya","ama","fakat","ile","için","üzere","göre","karşı","de","da","ki","bu","şu","o","bir","birçok",
    "çok","az","daha","en","olarak","ise","değil"
}
PUNCTS = set(list(",.;:!?()[]{}<>\"'“”‘’—–-/_\\|&*•"))
LANG_STOPS = {"EN": STOP_EN, "DE": STOP_DE, "TR": STOP_TR}

def rake_extract(text: str, stops: set, max_len=5, topn=10):
    if not text:
        return []
    tokens = re.split(r"(\W+)", text)
    phrases, buf = [], []
    for tok in tokens:
        t = tok.strip()
        if not t:
            continue
        low = t.lower()
        if low in stops or t in PUNCTS or re.fullmatch(r"\W+", t):
            if buf:
                phrases.append(" ".join(buf))
                buf = []
        else:
            buf.append(t)
    if buf:
        phrases.append(" ".join(buf))

    cleaned = []
    for ph in phrases:
        words = [w for w in ph.split() if not re.fullmatch(r"\W+", w)]
        if 1 <= len(words) <= max_len:
            cleaned.append(" ".join(words))
    if not cleaned:
        return []

    freq, degree = {}, {}
    for ph in cleaned:
        ws = ph.split()
        deg = sum(len(w) for w in ws)
        for w in ws:
            freq[w] = freq.get(w, 0) + 1
            degree[w] = degree.get(w, 0) + deg
    word_score = {w: degree[w] / max(freq[w], 1) for w in freq}
    scored = [(ph, sum(word_score.get(w, 0.0) for w in ph.split())) for ph in cleaned]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [f"{ph}:{sc:.3f}" for ph, sc in scored[:topn]]

feature_rows = []
tfidf_agg_lang_year = []
tfidf_agg_country_year = []

for lang in LANGS:
    df_lang = df_news[df_news["lang"] == lang].reset_index(drop=True)
    if df_lang.empty:
        continue

    print(f"[TF-IDF] {lang} — {len(df_lang)} articles")
    vec, X, docs, feats = build_tfidf_for_language(df_lang)

    with open(MODEL_DIR / f"tfidf_{lang}.pkl", "wb") as f:
        pickle.dump(vec, f)

    tfidf_json = [json.dumps(top_terms_from_row(X[i], feats), ensure_ascii=False) for i in range(X.shape[0])]
    stops = LANG_STOPS.get(lang, set())
    rake_json = [json.dumps(rake_extract(txt, stops=stops), ensure_ascii=False) for txt in docs]

    lang_feats = df_lang[["doc_id","lang","year","country","art_idx","len_tokens","len_chars"]].copy()
    lang_feats["tfidf_top_terms_json"] = tfidf_json
    lang_feats["rake_top_phrases_json"] = rake_json
    feature_rows.append(lang_feats)

    for year in sorted(df_lang["year"].unique()):
        mask = (df_lang["year"] == year).values
        X_sub = X[mask]
        col_sum = np.asarray(X_sub.sum(axis=0)).ravel()
        top_idx = np.argsort(col_sum)[::-1][:TOPN_TERMS_AGGR]
        top_terms = [f"{feats[j]}:{col_sum[j]:.4f}" for j in top_idx]
        tfidf_agg_lang_year.append({"lang": lang, "year": int(year),
                                    "top_terms_json": json.dumps(top_terms, ensure_ascii=False)})

    country_name = df_lang["country"].iloc[0]
    for year in sorted(df_lang["year"].unique()):
        mask = (df_lang["year"] == year).values
        X_sub = X[mask]
        col_sum = np.asarray(X_sub.sum(axis=0)).ravel()
        top_idx = np.argsort(col_sum)[::-1][:TOPN_TERMS_AGGR]
        top_terms = [f"{feats[j]}:{col_sum[j]:.4f}" for j in top_idx]
        tfidf_agg_country_year.append({"country": country_name, "year": int(year),
                                       "top_terms_json": json.dumps(top_terms, ensure_ascii=False)})

df_features_basic = pd.concat(feature_rows, ignore_index=True)
features_out = PROC_DIR / "features_basic.parquet"
df_features_basic.to_parquet(features_out, index=False)

df_tfidf_lang_year = pd.DataFrame(tfidf_agg_lang_year).sort_values(["lang","year"])
df_tfidf_ctry_year = pd.DataFrame(tfidf_agg_country_year).sort_values(["country","year"])
df_tfidf_lang_year.to_csv(TAB_DIR / "tfidf_top_terms_by_lang_year.csv", index=False, encoding="utf-8")
df_tfidf_ctry_year.to_csv(TAB_DIR / "tfidf_top_terms_by_country_year.csv", index=False, encoding="utf-8")

print("Saved:", features_out)
print("Saved:", TAB_DIR / "tfidf_top_terms_by_lang_year.csv", "|", TAB_DIR / "tfidf_top_terms_by_country_year.csv")

if 'append_log' in globals():
    append_log({
        "event": "features_basic_done",
        "rows": int(df_features_basic.shape[0]),
        "features_basic": str(features_out),
        "tfidf_lang_year_csv": str(TAB_DIR / "tfidf_top_terms_by_lang_year.csv"),
        "tfidf_country_year_csv": str(TAB_DIR / "tfidf_top_terms_by_country_year.csv"),
        "models": [str(MODEL_DIR / f"tfidf_{l}.pkl") for l in LANGS]
    })


[TF-IDF] EN — 1250 articles
[TF-IDF] DE — 1250 articles
[TF-IDF] TR — 1250 articles
Saved: /content/Project_Dissertation/data/processed/features_basic.parquet
Saved: /content/Project_Dissertation/artifacts/tables/tfidf_top_terms_by_lang_year.csv | /content/Project_Dissertation/artifacts/tables/tfidf_top_terms_by_country_year.csv


# Cell 9b — Quick Health Check

In [ ]:
# ============================================================
# Cell 9 — Quick Health Check (optional)
# ------------------------------------------------------------
# This cell checks if the TF-IDF and RAKE feature artifacts
# were successfully created. It verifies file presence,
# column structure, and basic counts per language.
# ============================================================

from pathlib import Path
import pandas as pd

ROOT_OUT = Path("/content/Project_Dissertation")
PROC_DIR  = ROOT_OUT / "data" / "processed"
TAB_DIR   = ROOT_OUT / "artifacts" / "tables"
MODEL_DIR = ROOT_OUT / "artifacts" / "models"

req_files = [
    PROC_DIR / "features_basic.parquet",
    MODEL_DIR / "tfidf_EN.pkl",
    MODEL_DIR / "tfidf_DE.pkl",
    MODEL_DIR / "tfidf_TR.pkl",
    TAB_DIR / "tfidf_top_terms_by_lang_year.csv",
    TAB_DIR / "tfidf_top_terms_by_country_year.csv",
]

print("== Artifact presence check ==")
ok = True
for p in req_files:
    status = "OK" if p.exists() else "MISSING"
    print(f"{status:8} {p}")
    ok = ok and p.exists()

if not ok:
    print("\nSome artifacts are missing. Re-run Cell 9 (feature extraction).")
else:
    df = pd.read_parquet(PROC_DIR / "features_basic.parquet")

    cols_needed = {
        "doc_id", "lang", "year", "country", "art_idx",
        "len_tokens", "len_chars",
        "tfidf_top_terms_json", "rake_top_phrases_json"
    }
    missing_cols = cols_needed - set(df.columns)
    print(f"\nRows: {len(df)}  |  Columns OK: {len(missing_cols) == 0}")
    if missing_cols:
        print("Missing columns:", missing_cols)

    print("\nCounts by language (approx. 1250 rows each if all 15 DOCX files exist):")
    print(df.groupby("lang").size())

    sample = df.sample(3, random_state=0)[
        ["lang", "art_idx", "tfidf_top_terms_json", "rake_top_phrases_json"]
    ]
    print("\nSample rows (top terms and keyphrases):")
    display(sample)


== Artifact presence check ==
OK       /content/Project_Dissertation/data/processed/features_basic.parquet
OK       /content/Project_Dissertation/artifacts/models/tfidf_EN.pkl
OK       /content/Project_Dissertation/artifacts/models/tfidf_DE.pkl
OK       /content/Project_Dissertation/artifacts/models/tfidf_TR.pkl
OK       /content/Project_Dissertation/artifacts/tables/tfidf_top_terms_by_lang_year.csv
OK       /content/Project_Dissertation/artifacts/tables/tfidf_top_terms_by_country_year.csv

Rows: 3750  |  Columns OK: True

Counts by language (approx. 1250 rows each if all 15 DOCX files exist):
lang
DE    1250
EN    1250
TR    1250
dtype: int64

Sample rows (top terms and keyphrases):


,lang,art_idx,tfidf_top_terms_json,rake_top_phrases_json
2250,DE,1,"[""Brandstiftung:0.3545"", ""Anklagebank:0.2571"",...","[""einem Sofa liegendes Zeitungspapier entzünde..."
1606,DE,107,"[""NUM aus:0.3436"", ""Zahl der:0.2386"", ""gesunke...","[""leistungsberechtigten Asylbewerber erneut ge..."
2836,TR,87,"[""gösterdiği:0.3019"", ""Fransa da:0.2759"", ""Muh...","[""Charlie Hebdo dergisini taşıyan eylemciler:1..."


# Cell 10 — Sentiment Analysis (EN/DE/TR)

In [ ]:
# ============================================================
# Cell 10 — Sentiment Analysis (EN/DE/TR)
# ------------------------------------------------------------
# This cell performs sentiment analysis using transformer models
# specific to each language. Sentiment scores are computed at the
# sentence level, averaged to article level, and then aggregated
# by year and country for comparative analysis.
#
# Outputs:
#   • /data/processed/sentiment_{EN,DE,TR}.parquet
#   • /data/processed/sentiment_all.parquet
#   • /artifacts/tables/sentiment_year_country_counts.csv
#   • /artifacts/tables/sentiment_year_country_probs.csv
# ============================================================

import os, re, json, numpy as np, pandas as pd, torch
from pathlib import Path

# --- Hugging Face cache (optional) ---
os.environ.setdefault("HF_HOME", "/content/drive/MyDrive/hf_cache")

# --- Ensure dependencies ---
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline
except Exception:
    %pip -q install transformers==4.43.3
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline

# --------------------------
# Configuration
# --------------------------
LANG_MODELS = {
    "EN": "cardiffnlp/twitter-roberta-base-sentiment-latest",  # 3-class (neg/neu/pos)
    "DE": "oliverguhr/german-sentiment-bert",                  # 3-class (neg/neu/pos)
    "TR": "savasy/bert-base-turkish-sentiment-cased"           # 2-class (neg/pos)
}
LANGS = ["EN", "DE", "TR"]
MAX_LEN = 128
MAX_SENTS = 12
BATCH_SIZE = 32
USE_GPU = torch.cuda.is_available()

# --------------------------
# Paths and Input
# --------------------------
ROOT_OUT = Path("/content/Project_Dissertation")
PROC_DIR = ROOT_OUT / "data" / "processed"
TAB_DIR = ROOT_OUT / "artifacts" / "tables"
PROC_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

if "df_news" not in globals():
    clean_path = PROC_DIR / "df_news_clean.parquet"
    assert clean_path.exists(), f"Missing dataset: {clean_path}"
    df_news = pd.read_parquet(clean_path)

# --------------------------
# Sentence Splitting
# --------------------------
_SENT_END_RE = re.compile(r"(?<=[.!?])\s+")

def split_sentences_fast(text: str, max_sents=MAX_SENTS):
    """Simple rule-based sentence segmentation."""
    text = (text or "").strip()
    if not text:
        return []
    sents = _SENT_END_RE.split(text)
    output = []
    for s in sents:
        s = s.strip()
        if not s:
            continue
        if len(s) > 600:
            output.extend([s[i:i+300] for i in range(0, len(s), 300)])
        else:
            output.append(s)
    return output[:max_sents]

# --------------------------
# Pipeline Handling
# --------------------------
_PIPE = {}

def get_pipeline(model_name: str):
    """Initialise or retrieve a cached Hugging Face pipeline."""
    if model_name in _PIPE:
        return _PIPE[model_name]
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_name)
    pipe = TextClassificationPipeline(
        model=mdl, tokenizer=tok,
        device=0 if USE_GPU else -1,
        truncation=True, max_length=MAX_LEN,
        top_k=None, return_all_scores=True
    )
    _PIPE[model_name] = (pipe, mdl.config.id2label)
    return _PIPE[model_name]

def unify_scores(all_scores, id2label):
    """
    Map model outputs to unified sentiment probabilities (neg, neu, pos).
    Supports both 3-class (neg/neu/pos) and 2-class (neg/pos) models.
    """
    p_neg = p_neu = p_pos = 0.0
    for item in all_scores:
        lab = str(item.get("label", "")).lower()
        p = float(item.get("score", 0.0))
        if "neg" in lab:
            p_neg += p
        elif "neu" in lab:
            p_neu += p
        elif "pos" in lab:
            p_pos += p
        else:
            try:
                idx = int(lab.split("_")[-1])
                name = str(id2label.get(idx, "")).lower()
                if "neg" in name:
                    p_neg += p
                elif "neu" in name:
                    p_neu += p
                elif "pos" in name:
                    p_pos += p
            except Exception:
                pass
    total = p_neg + p_neu + p_pos
    if total > 0:
        p_neg, p_neu, p_pos = p_neg / total, p_neu / total, p_pos / total
    return p_neg, p_neu, p_pos

# --------------------------
# Inference Loop (Per Language)
# --------------------------
article_blocks = []

for lang in LANGS:
    df_lang = df_news[df_news["lang"] == lang].copy()
    if df_lang.empty:
        continue

    model_name = LANG_MODELS[lang]
    pipe, id2label = get_pipeline(model_name)
    print(f"\nRunning sentiment inference for {lang} ({len(df_lang)} articles)...")

    records = []
    for _, r in df_lang.iterrows():
        text = f"{r['title_clean']}\n{r['body_clean']}"
        sents = split_sentences_fast(text)

        if not sents:
            p_neg = p_neu = p_pos = 0.0
            label, conf, n_scored = "NEU", 0.0, 0
        else:
            out = pipe(sents, batch_size=BATCH_SIZE)
            triples = [unify_scores(scores, id2label) for scores in out]
            p_neg = float(np.mean([t[0] for t in triples]))
            p_neu = float(np.mean([t[1] for t in triples]))
            p_pos = float(np.mean([t[2] for t in triples]))
            probs = np.array([p_neg, p_neu, p_pos])
            idx = int(np.argmax(probs))
            label = ["NEG", "NEU", "POS"][idx]
            conf = float(probs[idx])
            n_scored = len(sents)

        records.append({
            "doc_id": r["doc_id"], "lang": r["lang"], "year": int(r["year"]),
            "country": r["country"], "art_idx": int(r["art_idx"]),
            "n_sents_scored": n_scored,
            "p_neg": p_neg, "p_neu": p_neu, "p_pos": p_pos,
            "label": label, "conf": conf
        })

    df_sent = pd.DataFrame(records).sort_values(["lang", "year", "art_idx"]).reset_index(drop=True)
    out_path = PROC_DIR / f"sentiment_{lang}.parquet"
    df_sent.to_parquet(out_path, index=False)
    print("Saved:", out_path)
    article_blocks.append(df_sent)

# --------------------------
# Aggregate and Export Results
# --------------------------
df_all = pd.concat(article_blocks, ignore_index=True)
df_all.to_parquet(PROC_DIR / "sentiment_all.parquet", index=False)
print("Saved combined sentiment file:", PROC_DIR / "sentiment_all.parquet")

# Label proportions (NEG/NEU/POS) per year × country
counts = (
    df_all.groupby(["country", "year", "label"]).size()
    .reset_index(name="n")
)
totals = counts.groupby(["country", "year"])["n"].transform("sum")
counts["share"] = (counts["n"] / totals).round(4)
counts_pivot = (
    counts.pivot_table(index=["country", "year"], columns="label", values="share", fill_value=0)
    .reset_index()
)
counts_pivot.to_csv(TAB_DIR / "sentiment_year_country_counts.csv", index=False, encoding="utf-8")

# Average probabilities per year × country
probs = (
    df_all.groupby(["country", "year"], as_index=False)[["p_neg", "p_neu", "p_pos"]]
    .mean()
)
probs.to_csv(TAB_DIR / "sentiment_year_country_probs.csv", index=False, encoding="utf-8")

print("\nSaved:")
print(" • Sentiment counts:", TAB_DIR / "sentiment_year_country_counts.csv")
print(" • Sentiment probabilities:", TAB_DIR / "sentiment_year_country_probs.csv")

# --------------------------
# Optional Run Log
# --------------------------
if "append_log" in globals():
    append_log({
        "event": "sentiment_analysis_completed",
        "rows": int(df_all.shape[0]),
        "sentiment_all": str(PROC_DIR / "sentiment_all.parquet"),
        "counts_csv": str(TAB_DIR / "sentiment_year_country_counts.csv"),
        "probs_csv": str(TAB_DIR / "sentiment_year_country_probs.csv"),
        "models": LANG_MODELS,
        "device": "cuda" if USE_GPU else "cpu",
        "max_len": MAX_LEN, "max_sents": MAX_SENTS, "batch_size": BATCH_SIZE
    })

print("\nSentiment analysis completed successfully.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Device set to use cuda:0



Running sentiment inference for TR (1250 articles)...
Saved: /content/Project_Dissertation/data/processed/sentiment_TR.parquet
Saved combined sentiment file: /content/Project_Dissertation/data/processed/sentiment_all.parquet

Saved:
 • Sentiment counts: /content/Project_Dissertation/artifacts/tables/sentiment_year_country_counts.csv
 • Sentiment probabilities: /content/Project_Dissertation/artifacts/tables/sentiment_year_country_probs.csv

Sentiment analysis completed successfully.


# Cell 11 — Sentiment Summary and Sanity Report (Excel Export)

In [ ]:
# ============================================================
# Cell 11 — Sentiment Summary (final-final)
#   • Augmented shares+counts (printed + CSV)
#   • Avg probabilities rounded to 3 dp (printed + CSV)
#   • Excel (3 sheets) + ZIP refresh
# ============================================================

import pandas as pd
from pathlib import Path
import zipfile, os

ROOT_OUT = Path("/content/Project_Dissertation")
PROC_DIR = ROOT_OUT / "data" / "processed"
TAB_DIR  = ROOT_OUT / "artifacts" / "tables"
FIG_DIR  = ROOT_OUT / "artifacts" / "figures"
TAB_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# --- Load inputs ---
sent_all_path = PROC_DIR / "sentiment_all.parquet"
counts_csv    = TAB_DIR / "sentiment_year_country_counts.csv"   # shares (NEG/NEU/POS)
probs_csv     = TAB_DIR / "sentiment_year_country_probs.csv"    # avg p_neg/p_neu/p_pos

df_all  = pd.read_parquet(sent_all_path)
df_cnts = pd.read_csv(counts_csv)
df_prob = pd.read_csv(probs_csv)

# --- Build counts per country×year×label and merge (shares + counts) ---
label_col = "sent_label" if "sent_label" in df_all.columns else ("label" if "label" in df_all.columns else None)
if label_col is None:
    raise ValueError("No label column found (expected 'sent_label' or 'label').")

df_labels = df_all[["country","year",label_col]].copy()
df_labels["label_u"] = df_labels[label_col].astype(str).str.upper().map({
    "NEG":"NEG","NEU":"NEU","POS":"POS",
    "NEGATIVE":"NEG","NEUTRAL":"NEU","POSITIVE":"POS"
})

ct = (
    df_labels.groupby(["country","year","label_u"])
             .size().rename("n").reset_index()
    .pivot(index=["country","year"], columns="label_u", values="n")
    .fillna(0).astype(int).reset_index()
)
for k in ["NEG","NEU","POS"]:
    if k not in ct.columns:
        ct[k] = 0
ct["total_n"] = ct[["NEG","NEU","POS"]].sum(axis=1)

ct_counts = ct.rename(columns={"NEG":"NEG_n","NEU":"NEU_n","POS":"POS_n"})
df_cnts_aug = df_cnts.merge(ct_counts, on=["country","year"], how="left")
df_cnts_aug = df_cnts_aug[["country","year","NEG","NEU","POS","NEG_n","NEU_n","POS_n","total_n"]]

# --- Round average probabilities to 3 decimals ---
df_prob_rounded = df_prob.copy()
prob_cols = [c for c in df_prob_rounded.columns if c.startswith("p_")]
df_prob_rounded[prob_cols] = df_prob_rounded[prob_cols].round(3)

# --- PRINT the two key preview tables in the notebook output ---
print("Augmented label shares + counts (preview):")
display(df_cnts_aug.head(15))

print("\nRounded average probabilities (preview):")
display(df_prob_rounded.head(15))

# --- Save standalone CSVs so they appear in artifacts/tables ---
shares_aug_csv = TAB_DIR / "year_country_label_shares_augmented.csv"
df_cnts_aug.to_csv(shares_aug_csv, index=False, encoding="utf-8")

avg_probs_csv = TAB_DIR / "year_country_avg_probs_rounded.csv"
df_prob_rounded.to_csv(avg_probs_csv, index=False, encoding="utf-8")

counts_only_csv = TAB_DIR / "year_country_label_counts.csv"
ct_counts.to_csv(counts_only_csv, index=False, encoding="utf-8")

# --- Save canonical Excel with all three sheets ---
xlsx_path = TAB_DIR / "summary_tables.xlsx"
try:
    with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as xw:
        df_cnts_aug.to_excel(xw, sheet_name="year_country_label_shares", index=False)
        df_prob_rounded.to_excel(xw, sheet_name="year_country_avg_probs", index=False)
        ct_counts.to_excel(xw, sheet_name="year_country_label_counts", index=False)
        for sheet in ["year_country_label_shares","year_country_avg_probs","year_country_label_counts"]:
            ws = xw.sheets[sheet]
            ws.set_column(0, 0, 18)   # country
            ws.set_column(1, 1, 10)   # year
            ws.set_column(2, 20, 14)  # metrics
except Exception:
    with pd.ExcelWriter(xlsx_path) as xw:
        df_cnts_aug.to_excel(xw, sheet_name="year_country_label_shares", index=False)
        df_prob_rounded.to_excel(xw, sheet_name="year_country_avg_probs", index=False)
        ct_counts.to_excel(xw, sheet_name="year_country_label_counts", index=False)

# --- Refresh ZIP so it includes the updated Excel + CSVs ---
zip_path = ROOT_OUT / "NLP_artifacts.zip"

def zip_dir(dir_path: Path, zip_file: Path):
    with zipfile.ZipFile(zip_file, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(dir_path):
            for f in files:
                full = Path(root) / f
                rel  = full.relative_to(ROOT_OUT)
                zf.write(full, arcname=str(rel))

zip_dir(ROOT_OUT / "artifacts", zip_path)

# --- Artifact checklist ---
print("\n== Saved artifacts ==")
print("Excel:", xlsx_path)
print("Augmented shares CSV:", shares_aug_csv)
print("Rounded avg probs CSV:", avg_probs_csv)
print("Counts-only CSV:", counts_only_csv)
print("ZIP (refreshed):", zip_path)

# --- Optional run log ---
if 'append_log' in globals():
    append_log({
        "event": "sentiment_summary_final_export_v2",
        "excel": str(xlsx_path),
        "shares_aug_csv": str(shares_aug_csv),
        "avg_probs_csv": str(avg_probs_csv),
        "counts_only_csv": str(counts_only_csv),
        "zip": str(zip_path),
        "note": "Printed augmented shares + rounded probs; saved CSVs; ZIP updated."
    })


Augmented label shares + counts (preview):


,country,year,NEG,NEU,POS,NEG_n,NEU_n,POS_n,total_n
0,Germany,2019,0.020,0.980,0.000,5,245,0,250
1,Germany,2020,0.060,0.936,0.004,15,234,1,250
2,Germany,2021,0.112,0.884,0.004,28,221,1,250
3,Germany,2022,0.096,0.880,0.024,24,220,6,250
4,Germany,2023,0.040,0.956,0.004,10,239,1,250
5,Turkey,2019,0.124,0.000,0.876,31,0,219,250
6,Turkey,2020,0.668,0.000,0.332,167,0,83,250
7,Turkey,2021,0.740,0.000,0.260,185,0,65,250
8,Turkey,2022,0.720,0.000,0.280,180,0,70,250
9,Turkey,2023,0.772,0.000,0.228,193,0,57,250



Rounded average probabilities (preview):


,country,year,p_neg,p_neu,p_pos
0,Germany,2019,0.231,0.601,0.168
1,Germany,2020,0.251,0.572,0.177
2,Germany,2021,0.263,0.576,0.161
3,Germany,2022,0.274,0.545,0.180
4,Germany,2023,0.249,0.582,0.169
5,Turkey,2019,0.479,0.000,0.521
6,Turkey,2020,0.565,0.000,0.435
7,Turkey,2021,0.577,0.000,0.423
8,Turkey,2022,0.587,0.000,0.413
9,Turkey,2023,0.598,0.000,0.402



== Saved artifacts ==
Excel: /content/Project_Dissertation/artifacts/tables/summary_tables.xlsx
Augmented shares CSV: /content/Project_Dissertation/artifacts/tables/year_country_label_shares_augmented.csv
Rounded avg probs CSV: /content/Project_Dissertation/artifacts/tables/year_country_avg_probs_rounded.csv
Counts-only CSV: /content/Project_Dissertation/artifacts/tables/year_country_label_counts.csv
ZIP (refreshed): /content/Project_Dissertation/NLP_artifacts.zip


# Cell 12 — Visualization: Sentiment Shares and Average Probabilities

In [ ]:
# ============================================================
# Cell 12 — Visualization: Sentiment Shares and Average Probabilities
# ------------------------------------------------------------
# Generates figures and tables for thesis analysis.
# Outputs (saved in /artifacts/figures/ and /artifacts/tables/):
#   • Figures:
#       - Stacked bars: NEG/NEU/POS label shares by year for each country
#       - Lines: p_neg / p_neu / p_pos averages by year for each country
#   • Tables:
#       - sentiment_year_country_probs_z.csv (language-wise z-normalized)
# ============================================================

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

ROOT_OUT = Path("/content/Project_Dissertation")
PROC_DIR = ROOT_OUT / "data" / "processed"
TAB_DIR  = ROOT_OUT / "artifacts" / "tables"
FIG_DIR  = ROOT_OUT / "artifacts" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# --- Load summaries and article-level data ---
counts_csv = TAB_DIR / "sentiment_year_country_counts.csv"
probs_csv  = TAB_DIR / "sentiment_year_country_probs.csv"
df_counts  = pd.read_csv(counts_csv)
df_probs   = pd.read_csv(probs_csv)
df_all     = pd.read_parquet(PROC_DIR / "sentiment_all.parquet")

# --- 12A) Stacked bars: label shares by year per country ---
countries = df_counts["country"].unique().tolist()
FULL_LABELS = {"NEG": "Negative", "NEU": "Neutral", "POS": "Positive"}
COLORS      = {"Negative": "crimson", "Neutral": "gray", "Positive": "seagreen"}
ISO_MAP     = {"Germany":"DE", "Turkey":"TR", "United Kingdom":"UK"}

for c in countries:
    sub = df_counts[df_counts["country"] == c].sort_values("year").copy()
    for lab in ["NEG","NEU","POS"]:
        if lab not in sub.columns:
            sub[lab] = 0.0

    years = sub["year"].values
    x = np.arange(len(years))
    w = 0.25
    neg = sub["NEG"].values
    neu = sub["NEU"].values
    pos = sub["POS"].values

    fig, ax = plt.subplots(figsize=(8,4))
    ax.bar(x - w, neg, width=w, label="Negative", color=COLORS["Negative"])
    ax.bar(x     , neu, width=w, label="Neutral",  color=COLORS["Neutral"])
    ax.bar(x + w,  pos, width=w, label="Positive", color=COLORS["Positive"])

    ax.set_title(f"Label shares by year — {c}")
    ax.set_xlabel("Year")
    ax.set_ylabel("Share")
    ax.set_ylim(0, 1.0)
    ax.set_xticks(x); ax.set_xticklabels(years)   # X ekseni: sadece 2019..2023
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False, title="Sentiment")
    fig.tight_layout()

    iso = ISO_MAP[c]
    out = FIG_DIR / f"label_shares_{iso}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)



# --- 12B) Line plots: average probabilities by year per country ---
for c in countries:
    sub = df_probs[df_probs["country"] == c].sort_values("year").copy()
    years = sub["year"].astype(int).values

    plt.figure(figsize=(8, 4))
    plt.plot(years, sub["p_neg"].values, marker="o", label="p_neg")
    plt.plot(years, sub["p_neu"].values, marker="o", label="p_neu")
    plt.plot(years, sub["p_pos"].values, marker="o", label="p_pos")

    # enforce integer ticks (no .0 / .5)
    ax = plt.gca()
    ax.set_xticks(years)
    ax.set_xticklabels(years)
    ax.set_xlim(years.min() - 0.5, years.max() + 0.5)

    plt.title(f"Average probabilities by year — {c}")
    plt.xlabel("Year")
    plt.ylabel("Probability")
    plt.ylim(0, 1.0)
    plt.grid(alpha=0.3)
    plt.legend(frameon=False)
    plt.tight_layout()
    out = FIG_DIR / f"avg_probs_{c.replace(' ', '_')}.png"
    plt.savefig(out, dpi=150)
    plt.close()


# --- 12C) Language-wise z-normalized probabilities ---
# Z-scores computed within each language (article level), then averaged by country × year.
def zscore(x):
    x = np.asarray(x, dtype=float)
    mu, sd = np.mean(x), np.std(x)
    return (x - mu) / sd if sd > 0 else np.zeros_like(x, dtype=float)

df_z = df_all.copy()
for col in ["p_neg", "p_neu", "p_pos"]:
    df_z[col + "_z"] = df_z.groupby("lang")[col].transform(zscore)

probs_z = (
    df_z.groupby(["country", "year"], as_index=False)[["p_neg_z", "p_neu_z", "p_pos_z"]]
    .mean()
)
probs_z_out = TAB_DIR / "sentiment_year_country_probs_z.csv"
probs_z.to_csv(probs_z_out, index=False, encoding="utf-8")

# --- Plot z-normalized probability trends ---
for c in countries:
    sub = probs_z[probs_z["country"] == c].sort_values("year").copy()
    years = sub["year"].astype(int).values

    plt.figure(figsize=(8, 4))
    plt.plot(years, sub["p_neg_z"].values, marker="o", label="p_neg_z")
    plt.plot(years, sub["p_neu_z"].values, marker="o", label="p_neu_z")
    plt.plot(years, sub["p_pos_z"].values, marker="o", label="p_pos_z")

    # enforce integer ticks (no .0 / .5)
    ax = plt.gca()
    ax.set_xticks(years)
    ax.set_xticklabels(years)
    ax.set_xlim(years.min() - 0.5, years.max() + 0.5)

    plt.title(f"Z-normalized average probabilities by year — {c}")
    plt.xlabel("Year")
    plt.ylabel("z-score")
    plt.axhline(0, ls="--", lw=1, alpha=0.6, color="gray")
    plt.grid(alpha=0.3)
    plt.legend(frameon=False)
    plt.tight_layout()
    out = FIG_DIR / f"avg_probs_z_{c.replace(' ', '_')}.png"
    plt.savefig(out, dpi=150)
    plt.close()

print("Saved all figures to:", FIG_DIR)
print("Saved z-normalized table:", probs_z_out)

# --- Optional run log ---
if 'append_log' in globals():
    append_log({
        "event": "visualization_sentiment_done",
        "figures": [str(p) for p in FIG_DIR.glob("*.png")],
        "probs_z_csv": str(probs_z_out)
    })


Saved all figures to: /content/Project_Dissertation/artifacts/figures
Saved z-normalized table: /content/Project_Dissertation/artifacts/tables/sentiment_year_country_probs_z.csv


# Cell 13 — Final Dataset and Qualitative Exports

In [ ]:
# ============================================================
# Cell 13 — Final Dataset and Qualitative Exports
# ------------------------------------------------------------
# Builds a single analysis-ready table combining metadata,
# cleaned text, lexical features (TF-IDF/RAKE), and sentiment results.
# Also exports small qualitative CSV samples (per language)
# for inclusion in the thesis appendix.
#
# Outputs:
#   • /data/processed/df_news_final.parquet
#   • /artifacts/tables/article_examples_{EN,DE,TR}.csv
# ============================================================

import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

# --- Paths ---
ROOT_OUT = Path("/content/Project_Dissertation")
PROC_DIR = ROOT_OUT / "data" / "processed"
TAB_DIR = ROOT_OUT / "artifacts" / "tables"
PROC_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

# --- Load core inputs ---
df_news  = pd.read_parquet(PROC_DIR / "df_news_clean.parquet")       # cleaned articles
df_basic = pd.read_parquet(PROC_DIR / "features_basic.parquet")      # TF-IDF / RAKE
df_sent  = pd.read_parquet(PROC_DIR / "sentiment_all.parquet")       # sentiment results

# Optional morphosyntactic features (if available)
morph_path = PROC_DIR / "features_morph.parquet"
df_morph = pd.read_parquet(morph_path) if morph_path.exists() else None

# --- Ensure key consistency ---
KEYS = ["doc_id", "lang", "year", "country", "art_idx"]
for df in (df_news, df_basic, df_sent, *( [df_morph] if df_morph is not None else [] )):
    df["year"] = df["year"].astype(int)
    df["art_idx"] = df["art_idx"].astype(int)

# --- Merge datasets ---
text_cols = ["title_raw", "body_raw", "title_clean", "body_clean", "len_tokens", "len_chars"]
news_cols = KEYS + [c for c in text_cols if c in df_news.columns]
basic_cols = KEYS + ["tfidf_top_terms_json", "rake_top_phrases_json"]
sent_cols  = KEYS + ["n_sents_scored", "p_neg", "p_neu", "p_pos", "label", "conf"]

df_final = (
    df_news[news_cols]
    .merge(df_basic[basic_cols], on=KEYS, how="left")
    .merge(df_sent[sent_cols], on=KEYS, how="left")
)

if df_morph is not None:
    morph_keep = [c for c in df_morph.columns if c not in df_final.columns]
    df_final = df_final.merge(df_morph[morph_keep + KEYS], on=KEYS, how="left")

# --- Save merged dataset ---
final_path = PROC_DIR / "df_news_final.parquet"
df_final.to_parquet(final_path, index=False)
print("Saved final dataset:", final_path)
print("Rows:", len(df_final), "| Columns:", len(df_final.columns))

# --- Qualitative examples for appendix ---
N_PER_LABEL_PER_YEAR = 10
LANGS = ["EN", "DE", "TR"]

def first_n(s, n=300):
    s = (s or "").strip()
    return s[:n].replace("\n", " ")

for lang in LANGS:
    sub = df_final[df_final["lang"] == lang].copy()
    if sub.empty:
        continue
    rows = []
    for year in sorted(sub["year"].unique()):
        for lab in ["NEG", "NEU", "POS"]:
            mask = (sub["year"] == year) & (sub["label"] == lab)
            if mask.sum() == 0:
                continue
            bucket = sub[mask].sample(n=min(N_PER_LABEL_PER_YEAR, mask.sum()), random_state=42)
            for _, r in bucket.iterrows():
                rows.append({
                    "doc_id": r["doc_id"],
                    "lang": r["lang"],
                    "year": int(r["year"]),
                    "country": r["country"],
                    "art_idx": int(r["art_idx"]),
                    "label": r["label"],
                    "conf": round(float(r["conf"]), 3) if pd.notna(r["conf"]) else None,
                    "title_clean": r.get("title_clean", ""),
                    "body_preview": first_n(r.get("body_clean", ""), 300)
                })
    df_examples = pd.DataFrame(rows)
    out_csv = TAB_DIR / f"article_examples_{lang}.csv"
    df_examples.to_csv(out_csv, index=False, encoding="utf-8")
    print(f"Saved qualitative examples ({lang}):", out_csv)

# --- Optional run log ---
if 'append_log' in globals():
    append_log({
        "event": "data_products_written",
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "final_path": str(final_path),
        "examples": {lang: str(TAB_DIR / f"article_examples_{lang}.csv") for lang in LANGS}
    })


Saved final dataset: /content/Project_Dissertation/data/processed/df_news_final.parquet
Rows: 3750 | Columns: 19
Saved qualitative examples (EN): /content/Project_Dissertation/artifacts/tables/article_examples_EN.csv
Saved qualitative examples (DE): /content/Project_Dissertation/artifacts/tables/article_examples_DE.csv
Saved qualitative examples (TR): /content/Project_Dissertation/artifacts/tables/article_examples_TR.csv


# Cell 14 — Reproducibility and Audit Trail

In [ ]:
# ============================================================
# Cell 14 — Reproducibility and Audit Trail
# ------------------------------------------------------------
# Records key information for reproducibility:
#   • Environment and package versions (requirements_versions.txt)
#   • SHA-1 hashes of inputs and outputs (io_hashes.csv)
#   • Hardware, global seeds, and run metadata (run_log.jsonl)
# ============================================================

import os, sys, json, hashlib, platform, subprocess, random, time
from datetime import datetime, timezone
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

try:
    import torch
except Exception:
    torch = None

# --- Paths ---
ROOT_OUT = Path("/content/Project_Dissertation")
PROC_DIR = ROOT_OUT / "data" / "processed"
ART_DIR  = ROOT_OUT / "artifacts"
TAB_DIR  = ART_DIR / "tables"
FIG_DIR  = ART_DIR / "figures"
LOG_DIR  = ART_DIR / "logs"
for d in [PROC_DIR, TAB_DIR, FIG_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REQ_TXT   = LOG_DIR / "requirements_versions.txt"
IO_HASHES = TAB_DIR / "io_hashes.csv"
RUN_LOG   = LOG_DIR / "run_log.jsonl"

# --- 1) Global seeds ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if torch is not None:
    try:
        torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)
    except Exception:
        pass

# --- 2) Environment info ---
def safe_version(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "N/A"

pkgs = [
    "python", "pip", "numpy", "pandas", "pyarrow",
    "transformers", "torch", "stanza", "scikit-learn",
    "rake-nltk", "langdetect", "python-docx", "docx2txt", "matplotlib"
]

rows = []
rows.append(f"timestamp_utc = {datetime.now(timezone.utc).isoformat()}")
rows.append(f"python = {platform.python_version()}")
rows.append(f"platform = {platform.platform()}")
rows.append(f"machine = {platform.machine()}")
rows.append(f"processor = {platform.processor()}")

gpu_name = "None"
if torch is not None and torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
rows.append(f"gpu_available = {bool(torch and torch.cuda.is_available())}")
rows.append(f"gpu_name = {gpu_name}")

# RAM info (optional)
try:
    import psutil
except Exception:
    psutil = None
if psutil:
    vm = psutil.virtual_memory()
    rows.append(f"ram_total_gb = {round(vm.total / 1e9, 2)}")
else:
    rows.append("ram_total_gb = N/A")

# Package versions
rows.append("\n# Package versions")
for p in pkgs:
    if p == "python":
        continue
    rows.append(f"{p} == {safe_version(p)}")

REQ_TXT.write_text("\n".join(rows), encoding="utf-8")
print("Saved:", REQ_TXT)

# --- 3) SHA-1 helper ---
def sha1_of_file(path: Path, chunk_size=1 << 20):
    try:
        h = hashlib.sha1()
        with open(path, "rb") as f:
            while True:
                b = f.read(chunk_size)
                if not b:
                    break
                h.update(b)
        return h.hexdigest()
    except Exception:
        return None

# --- 4) Enumerate files for hashing ---
DATA_DIR = Path("/content/drive/MyDrive/Project Dissertation/AI_Refugee_Project/data/unstructured/DATASET")
docx_files = []
for folder in ["German", "Turkey", "United Kingdom"]:
    root = DATA_DIR / folder
    if root.exists():
        docx_files += sorted(root.glob("*.docx"))

processed_files = [
    PROC_DIR / "df_news_raw.parquet",
    PROC_DIR / "df_news_clean.parquet",
    PROC_DIR / "features_basic.parquet",
    PROC_DIR / "features_morph.parquet",
    PROC_DIR / "sentiment_EN.parquet",
    PROC_DIR / "sentiment_DE.parquet",
    PROC_DIR / "sentiment_TR.parquet",
    PROC_DIR / "sentiment_all.parquet",
    PROC_DIR / "df_news_final.parquet",
]

table_files = [
    TAB_DIR / "segmentation_counts.csv",
    TAB_DIR / "language_mismatch_summary.csv",
    TAB_DIR / "tfidf_top_terms_by_lang_year.csv",
    TAB_DIR / "tfidf_top_terms_by_country_year.csv",
    TAB_DIR / "sentiment_year_country_counts.csv",
    TAB_DIR / "sentiment_year_country_probs.csv",
    TAB_DIR / "sentiment_year_country_probs_z.csv",
    TAB_DIR / "article_examples_EN.csv",
    TAB_DIR / "article_examples_DE.csv",
    TAB_DIR / "article_examples_TR.csv",
]

figure_files = list(FIG_DIR.glob("*.png"))

all_targets = [*docx_files, *processed_files, *table_files, *figure_files]
all_targets = [p for p in all_targets if p is not None and Path(p).exists()]

# --- 5) Compute and save hashes ---
hash_rows = []
for p in all_targets:
    h = sha1_of_file(Path(p))
    hash_rows.append({"path": str(p), "sha1": h})

df_hash = pd.DataFrame(hash_rows).sort_values("path").reset_index(drop=True)
df_hash.to_csv(IO_HASHES, index=False, encoding="utf-8")
print("Saved:", IO_HASHES)

# --- 6) Append run log ---
run_entry = {
    "event": "repro_audit_dump",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "gpu_available": bool(torch and torch.cuda.is_available()),
    "gpu_name": gpu_name,
    "rows_hashed": int(len(df_hash)),
    "requirements": str(REQ_TXT),
    "hash_table": str(IO_HASHES)
}
with open(RUN_LOG, "a", encoding="utf-8") as f:
    f.write(json.dumps(run_entry, ensure_ascii=False) + "\n")

print("Appended run log:", RUN_LOG)


Saved: /content/Project_Dissertation/artifacts/logs/requirements_versions.txt
Saved: /content/Project_Dissertation/artifacts/tables/io_hashes.csv
Appended run log: /content/Project_Dissertation/artifacts/logs/run_log.jsonl


# Cell 15 — Risk Controls and Soft Quality Checks

In [ ]:
# ============================================================
# Cell 15 — Risk Controls and Soft Quality Checks
# ------------------------------------------------------------
# Performs integrity checks on core outputs:
#   • Ensures all key artifacts exist.
#   • Checks for missing text, duplicate keys, and article counts.
#   • Validates sentiment probabilities and identifies extreme label shares.
#   • Saves review tables for manual inspection instead of raising errors.
#
# Outputs (saved to /artifacts/tables/):
#   • qc_missing_text.csv
#   • qc_article_counts_by_doc.csv
#   • qc_duplicate_keys.csv
#   • qc_prob_sum_issues.csv
#   • qc_extreme_label_shares.csv
# ============================================================

import os
import numpy as np
import pandas as pd
from pathlib import Path

# --- Paths ---
ROOT_OUT = Path("/content/Project_Dissertation")
PROC_DIR = ROOT_OUT / "data" / "processed"
ART_DIR  = ROOT_OUT / "artifacts"
TAB_DIR  = ART_DIR / "tables"
FIG_DIR  = ART_DIR / "figures"
LOG_DIR  = ART_DIR / "logs"
for d in [PROC_DIR, TAB_DIR, FIG_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- 1) Check existence of main artifacts ---
must_exist = [
    PROC_DIR / "df_news_clean.parquet",
    PROC_DIR / "features_basic.parquet",
    PROC_DIR / "sentiment_all.parquet",
    PROC_DIR / "df_news_final.parquet",
    TAB_DIR  / "sentiment_year_country_counts.csv",
    TAB_DIR  / "sentiment_year_country_probs.csv",
]
missing = [str(p) for p in must_exist if not Path(p).exists()]
if missing:
    print("[WARN] Missing artifacts:")
    for m in missing:
        print("  -", m)
else:
    print("[ok] All key artifacts exist.")

# --- 2) Load core datasets ---
df_clean = pd.read_parquet(PROC_DIR / "df_news_clean.parquet")
df_sent  = pd.read_parquet(PROC_DIR / "sentiment_all.parquet")
df_final = pd.read_parquet(PROC_DIR / "df_news_final.parquet")

# --- 3) Missing text (title/body) ---
mask_missing = (
    (df_clean["title_clean"].isna() | (df_clean["title_clean"].str.strip() == "")) |
    (df_clean["body_clean"].isna()  | (df_clean["body_clean"].str.strip() == ""))
)
df_missing = df_clean.loc[mask_missing, ["doc_id","lang","year","country","art_idx","title_clean","body_clean"]]
if not df_missing.empty:
    out = TAB_DIR / "qc_missing_text.csv"
    df_missing.to_csv(out, index=False, encoding="utf-8")
    print(f"[WARN] Missing or empty text rows: {len(df_missing)}  -> {out}")
else:
    print("[ok] No missing title/body text.")

# --- 4) Article counts per document ---
counts = (
    df_clean.groupby(["doc_id","lang","year","country"])
            .size().reset_index(name="n_articles")
)
out_counts = TAB_DIR / "qc_article_counts_by_doc.csv"
counts.to_csv(out_counts, index=False, encoding="utf-8")
zero_docs = counts[counts["n_articles"] == 0]
if not zero_docs.empty:
    print(f"[WARN] Documents with zero articles: {len(zero_docs)}")
else:
    print("[ok] All documents contain at least one article.")
print("Saved:", out_counts)

# --- 5) Duplicate (doc_id, art_idx) check ---
dup_mask = df_clean.duplicated(subset=["doc_id","art_idx"], keep=False)
df_dups = df_clean.loc[dup_mask, ["doc_id","lang","year","country","art_idx"]].sort_values(["doc_id","art_idx"])
if not df_dups.empty:
    out = TAB_DIR / "qc_duplicate_keys.csv"
    df_dups.to_csv(out, index=False, encoding="utf-8")
    print(f"[WARN] Duplicate (doc_id, art_idx) entries: {len(df_dups)}  -> {out}")
else:
    print("[ok] No duplicate (doc_id, art_idx) keys found.")

# --- 6) Sentiment probability validation ---
def _prob_issue(row, tol=0.01):
    pn, pu, pp = float(row["p_neg"]), float(row["p_neu"]), float(row["p_pos"])
    if not (0 <= pn <= 1 and 0 <= pu <= 1 and 0 <= pp <= 1):
        return True
    s = pn + pu + pp
    if (pn == 0 and pu == 0 and pp == 0):
        return False  # allowed for empty sentences
    return not (1 - tol <= s <= 1 + tol)

prob_issues = df_sent[df_sent.apply(_prob_issue, axis=1)]
if not prob_issues.empty:
    out = TAB_DIR / "qc_prob_sum_issues.csv"
    prob_issues.to_csv(out, index=False, encoding="utf-8")
    print(f"[WARN] Probability sum/range issues: {len(prob_issues)}  -> {out}")
else:
    print("[ok] Probability triples are within valid range.")

# --- 7) Extreme label shares by country × year ---
shares = pd.read_csv(TAB_DIR / "sentiment_year_country_counts.csv")
ext_rows = []
for _, g in shares.groupby(["country","year"], as_index=False):
    c, y = g["country"].iloc[0], int(g["year"].iloc[0])
    row = {"country": c, "year": y}
    for lab in ["NEG","NEU","POS"]:
        row[lab] = float(g[lab].iloc[0]) if lab in g.columns else 0.0
    flags = []
    for lab in ["NEG","NEU","POS"]:
        v = row[lab]
        if c == "Turkey" and lab == "NEU":  # expected ≈0 for TR
            continue
        if v >= 0.98 or v <= 0.02:
            flags.append(f"{lab}={v:.3f}")
    if flags:
        row["flags"] = "; ".join(flags)
        ext_rows.append(row)

df_ext = pd.DataFrame(ext_rows)
if not df_ext.empty:
    out = TAB_DIR / "qc_extreme_label_shares.csv"
    df_ext.to_csv(out, index=False, encoding="utf-8")
    print(f"[INFO] Extreme label shares found: {len(df_ext)}  -> {out}")
else:
    print("[ok] No extreme label shares (TR-NEU excluded by design).")

# --- 8) Compact QC summary ---
print("\n== QC Summary ==")
print("Articles (clean):", len(df_clean))
print("Sentiment rows :", len(df_sent))
print("Merged rows    :", len(df_final))
print("Unique documents:", df_clean["doc_id"].nunique())

# --- 9) Optional run log ---
if 'append_log' in globals():
    append_log({
        "event": "qc_soft_checks_done",
        "missing_text_rows": int(df_missing.shape[0]) if not df_missing.empty else 0,
        "docs_with_zero_articles": int(zero_docs.shape[0]) if not zero_docs.empty else 0,
        "duplicate_key_rows": int(df_dups.shape[0]) if not df_dups.empty else 0,
        "prob_issue_rows": int(prob_issues.shape[0]) if not prob_issues.empty else 0,
        "extreme_share_rows": int(df_ext.shape[0]) if not df_ext.empty else 0
    })


[ok] All key artifacts exist.
[ok] No missing title/body text.
[ok] All documents contain at least one article.
Saved: /content/Project_Dissertation/artifacts/tables/qc_article_counts_by_doc.csv
[ok] No duplicate (doc_id, art_idx) keys found.
[ok] Probability triples are within valid range.
[INFO] Extreme label shares found: 9  -> /content/Project_Dissertation/artifacts/tables/qc_extreme_label_shares.csv

== QC Summary ==
Articles (clean): 3750
Sentiment rows : 3750
Merged rows    : 3750
Unique documents: 15


# Cell 16 — Hypothesis Tests and Robustness Analysis

In [ ]:
# ============================================================
# Cell 16 — Hypothesis Tests and Robustness Analysis
# ------------------------------------------------------------
# This cell performs the main evaluation for the thesis:
#   1) Year-over-year sentiment trends per country (OLS with HAC SEs)
#   2) Cross-country sentiment differences (ANOVA, Kruskal–Wallis, post-hoc)
#   3) Robustness check using confidence thresholding (Cohen’s κ)
#
# Outputs (in /artifacts/tables/):
#   • trend_results.csv
#   • anova_table.csv
#   • kruskal_table.csv
#   • posthoc_pairwise.csv
#   • robustness_kappa.csv
# ============================================================

import numpy as np, pandas as pd
from pathlib import Path

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from scipy.stats import kruskal, mannwhitneyu
from sklearn.metrics import cohen_kappa_score

# --- Paths ---
ROOT_OUT = Path("/content/Project_Dissertation")
PROC_DIR = ROOT_OUT / "data" / "processed"
TAB_DIR  = ROOT_OUT / "artifacts" / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

df_all = pd.read_parquet(PROC_DIR / "sentiment_all.parquet").copy()

# --- 0) Continuous sentiment score ---
df_all["sent_score"] = df_all["p_pos"] - df_all["p_neg"]

# --- 1) Year-over-year trend per country (OLS + HAC SEs) ---
def ols_hac_trend(df_c):
    g = (df_c.groupby("year", as_index=False)["sent_score"]
               .mean().rename(columns={"sent_score":"y"}))
    g = g.sort_values("year").reset_index(drop=True)
    g["t"] = g["year"] - g["year"].min()
    X = sm.add_constant(g["t"])
    y = g["y"]
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 1})
    slope = model.params["t"]
    conf_low, conf_high = model.conf_int().loc["t"].tolist()
    pval = model.pvalues["t"]
    return {
        "n_years": int(g.shape[0]),
        "slope_per_year": float(slope),
        "ci_low": float(conf_low),
        "ci_high": float(conf_high),
        "p_value": float(pval)
    }

trend_rows = []
for country in sorted(df_all["country"].unique()):
    res = ols_hac_trend(df_all[df_all["country"] == country])
    res.update({"country": country})
    trend_rows.append(res)

df_trend = (
    pd.DataFrame(trend_rows)
    .loc[:, ["country", "n_years", "slope_per_year", "ci_low", "ci_high", "p_value"]]
    .sort_values("country")
)
trend_out = TAB_DIR / "trend_results.csv"
df_trend.to_csv(trend_out, index=False, encoding="utf-8")
print("Saved:", trend_out)

# --- 2) Cross-country differences ---
# (a) One-way ANOVA
model = smf.ols("sent_score ~ C(country)", data=df_all).fit()
anova_tbl = anova_lm(model, typ=2)
df_anova = anova_tbl.reset_index().rename(columns={"index": "term"})
anova_out = TAB_DIR / "anova_table.csv"
df_anova.to_csv(anova_out, index=False, encoding="utf-8")
print("Saved:", anova_out)

# (b) Kruskal–Wallis test
groups = [df_all.loc[df_all["country"] == c, "sent_score"].values
          for c in sorted(df_all["country"].unique())]
kw_stat, kw_p = kruskal(*groups)
df_kw = pd.DataFrame([{"test": "Kruskal–Wallis", "H": kw_stat, "p_value": kw_p}])
kw_out = TAB_DIR / "kruskal_table.csv"
df_kw.to_csv(kw_out, index=False, encoding="utf-8")
print("Saved:", kw_out)

# (c) Pairwise Mann–Whitney U tests + Holm correction + Cliff’s delta
def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    allv = np.concatenate([x, y])
    ranks = pd.Series(allv).rank(method="average").values
    rx = ranks[:nx].sum()
    ry = ranks[nx:].sum()
    Ux = rx - nx * (nx + 1) / 2.0
    Uy = ry - ny * (ny + 1) / 2.0
    delta = (Ux - Uy) / (nx * ny)
    return float(delta)

countries = sorted(df_all["country"].unique())
pairs, pvals, deltas, n_x, n_y = [], [], [], [], []
for i in range(len(countries)):
    for j in range(i + 1, len(countries)):
        c1, c2 = countries[i], countries[j]
        x = df_all.loc[df_all["country"] == c1, "sent_score"].values
        y = df_all.loc[df_all["country"] == c2, "sent_score"].values
        U, p = mannwhitneyu(x, y, alternative="two-sided")
        d = cliffs_delta(x, y)
        pairs.append((c1, c2))
        pvals.append(p)
        deltas.append(d)
        n_x.append(len(x))
        n_y.append(len(y))

pvals = np.array(pvals, dtype=float)
m = len(pvals)
order = np.argsort(pvals)
adj = np.empty_like(pvals)
prev = 0.0
for k, idx in enumerate(order):
    adj_val = (m - k) * pvals[idx]
    adj[idx] = max(adj_val, prev)
    prev = adj[idx]
adj = np.clip(adj, 0, 1)

df_posthoc = pd.DataFrame({
    "country_1": [a for a, _ in pairs],
    "country_2": [b for _, b in pairs],
    "n1": n_x, "n2": n_y,
    "p_raw": pvals,
    "p_holm": adj,
    "cliffs_delta": deltas
}).sort_values("p_holm")
posthoc_out = TAB_DIR / "posthoc_pairwise.csv"
df_posthoc.to_csv(posthoc_out, index=False, encoding="utf-8")
print("Saved:", posthoc_out)

# --- 3) Robustness test (Cohen’s κ) ---
def thresholded_label(row, thr=0.55):
    conf = float(row["conf"])
    return "NEU" if conf < thr else row["label"]

df_all["label_thr"] = df_all.apply(thresholded_label, axis=1)

rows = []
rows.append({
    "scope": "overall",
    "n": int(df_all.shape[0]),
    "kappa": float(cohen_kappa_score(df_all["label"], df_all["label_thr"]))
})
for lang in sorted(df_all["lang"].unique()):
    sub = df_all[df_all["lang"] == lang]
    rows.append({
        "scope": f"lang={lang}",
        "n": int(sub.shape[0]),
        "kappa": float(cohen_kappa_score(sub["label"], sub["label_thr"]))
    })

df_kappa = pd.DataFrame(rows)
kappa_out = TAB_DIR / "robustness_kappa.csv"
df_kappa.to_csv(kappa_out, index=False, encoding="utf-8")
print("Saved:", kappa_out)

# --- Console summary ---
print("\n== Yearly trends ==")
display(df_trend)

print("\n== ANOVA (sent_score ~ country) ==")
display(df_anova)

print("\n== Kruskal–Wallis ==")
display(df_kw)

print("\n== Pairwise post-hoc comparisons ==")
display(df_posthoc.head(10))

print("\n== Robustness (Cohen’s κ) ==")
display(df_kappa)


Saved: /content/Project_Dissertation/artifacts/tables/trend_results.csv
Saved: /content/Project_Dissertation/artifacts/tables/anova_table.csv
Saved: /content/Project_Dissertation/artifacts/tables/kruskal_table.csv
Saved: /content/Project_Dissertation/artifacts/tables/posthoc_pairwise.csv
Saved: /content/Project_Dissertation/artifacts/tables/robustness_kappa.csv

== Yearly trends ==


,country,n_years,slope_per_year,ci_low,ci_high,p_value
0,Germany,5,-0.005257,-0.011237,0.000723,0.084870
1,Turkey,5,-0.052083,-0.075884,-0.028282,0.000018
2,United Kingdom,5,-0.000770,-0.012796,0.011257,0.900199



== ANOVA (sent_score ~ country) ==


,term,sum_sq,df,F,PR(>F)
0,C(country),1.757443,2.0,18.362087,1.159505e-08
1,Residual,179.313501,3747.0,NaN,NaN



== Kruskal–Wallis ==


,test,H,p_value
0,Kruskal–Wallis,86.66139,1.519562e-19



== Pairwise post-hoc comparisons ==


,country_1,country_2,n1,n2,p_raw,p_holm,cliffs_delta
1,Germany,United Kingdom,1250,1250,2.873599e-30,8.620796e-30,0.264078
0,Germany,Turkey,1250,1250,4.840781e-03,9.681562e-03,0.065060
2,Turkey,United Kingdom,1250,1250,2.621239e-02,2.621239e-02,0.051334



== Robustness (Cohen’s κ) ==


,scope,n,kappa
0,overall,3750,0.706141
1,lang=DE,1250,0.582868
2,lang=EN,1250,0.579367
3,lang=TR,1250,0.453856


# Cell 17 — Publication-Grade Visual Report (Trends & Differences)

In [ ]:
# ============================================================
# Cell 17 — Publication-Grade Visual Report (Trends & Differences)
# ------------------------------------------------------------
# Produces final publication-quality figures and captions:
#   • Higher DPI (300) and vector PDF export
#   • Consistent y-axis limits across countries
#   • Thesis-ready titles and caption CSV
# ============================================================

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

ROOT_OUT = Path("/content/Project_Dissertation")
PROC_DIR = ROOT_OUT / "data" / "processed"
TAB_DIR  = ROOT_OUT / "artifacts" / "tables"
FIG_DIR  = ROOT_OUT / "artifacts" / "figures"
CAP_DIR  = ROOT_OUT / "artifacts" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
CAP_DIR.mkdir(parents=True, exist_ok=True)

# --- Load data ---
df_all   = pd.read_parquet(PROC_DIR / "sentiment_all.parquet").copy()
df_trend = pd.read_csv(TAB_DIR / "trend_results.csv")
df_pair  = pd.read_csv(TAB_DIR / "posthoc_pairwise.csv")

# Continuous sentiment score
df_all["sent_score"] = df_all["p_pos"] - df_all["p_neg"]
countries = sorted(df_all["country"].unique().tolist())

# --- Plot style ---
FIG_DPI = 300
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

# --- Compute global y-limits (for consistent scaling) ---
mins, maxs = [], []
for c in countries:
    sub = df_all[df_all["country"] == c]
    g = (
        sub.groupby("year")["sent_score"]
           .agg(mean="mean", sem=lambda x: x.std(ddof=1)/np.sqrt(len(x)))
           .reset_index()
    )
    ci = 1.96 * g["sem"].values
    mins.append((g["mean"].values - ci).min())
    maxs.append((g["mean"].values + ci).max())
ymin = float(min(mins)) - 0.02
ymax = float(max(maxs)) + 0.02

# --- A) Trend plots (yearly mean ± 95% CI) ---
captions = []
for c in countries:
    sub = df_all[df_all["country"] == c]
    g = (
        sub.groupby("year")["sent_score"]
           .agg(mean="mean", sem=lambda x: x.std(ddof=1)/np.sqrt(len(x)))
           .reset_index()
    )
    g["ci"] = 1.96 * g["sem"]

    years = g["year"].values
    means, cis = g["mean"].values, g["ci"].values

    plt.figure(figsize=(7.5, 4.2))
    plt.errorbar(years, means, yerr=cis, fmt="o-", capsize=4)
    plt.axhline(0, ls="--", lw=1, alpha=0.6)
    plt.ylim(ymin, ymax)
    plt.title(f"Yearly sentiment trend — {c}")
    plt.xlabel("Year")
    plt.ylabel("Mean sentiment score (p_pos − p_neg)")
    plt.grid(alpha=0.3)

    # Annotate slope and p-value
    row = df_trend[df_trend["country"] == c].iloc[0]
    slope, ci_lo, ci_hi, p = row["slope_per_year"], row["ci_low"], row["ci_high"], row["p_value"]
    note = f"slope/year = {slope:.3f}, 95% CI [{ci_lo:.3f}, {ci_hi:.3f}], p={p:.3g}"
    plt.gcf().text(0.02, -0.10, note, fontsize=9)

    # after plotting:
    ax = plt.gca()
    ax.set_xticks(years.astype(int))         # show only 2019..2023
    ax.set_xticklabels(years.astype(int))    # ensure clean integer tick labels
    ax.set_xlim(years.min() - 0.5, years.max() + 0.5)



    # Save figures
    base = f"trend_sent_score_{c.replace(' ', '_')}"
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{base}.png", dpi=FIG_DPI, bbox_inches="tight")
    plt.savefig(FIG_DIR / f"{base}.pdf", dpi=FIG_DPI, bbox_inches="tight", transparent=True)
    plt.close()

    captions.append({
        "figure": f"{base}.pdf",
        "caption": f"Yearly mean sentiment score (p_pos − p_neg) for {c}, "
                   f"with 95% CI. OLS (HAC) slope per year = {slope:.3f} "
                   f"(95% CI [{ci_lo:.3f}, {ci_hi:.3f}], p={p:.3g})."
    })

# --- B) Violin plot: sentiment distribution by country ---
data = [df_all.loc[df_all["country"] == c, "sent_score"].values for c in countries]
plt.figure(figsize=(8, 4.6))
plt.violinplot(data, showmeans=False, showmedians=False, showextrema=False)
xpos = np.arange(1, len(countries) + 1)
means = [np.mean(d) for d in data]
sems  = [np.std(d, ddof=1) / np.sqrt(len(d)) for d in data]
plt.errorbar(xpos, means, yerr=[1.96 * s for s in sems], fmt="o", capsize=4)
plt.axhline(0, ls="--", lw=1, alpha=0.6)
plt.xticks(xpos, countries)
plt.ylabel("Sentiment score (p_pos − p_neg)")
plt.title("Distribution of sentiment scores by country")
plt.grid(alpha=0.25, axis="y")
plt.tight_layout()
plt.savefig(FIG_DIR / "violin_sent_score_by_country.png", dpi=FIG_DPI, bbox_inches="tight")
plt.savefig(FIG_DIR / "violin_sent_score_by_country.pdf", dpi=FIG_DPI, bbox_inches="tight", transparent=True)
plt.close()

captions.append({
    "figure": "violin_sent_score_by_country.pdf",
    "caption": "Distribution of article-level sentiment scores by country. "
               "Points indicate mean values with 95% confidence intervals."
})

# --- C) Cliff’s delta bar plot (pairwise country differences) ---
pairs_order = [("Germany", "United Kingdom"),
               ("Germany", "Turkey"),
               ("Turkey", "United Kingdom")]
rows = []
for a, b in pairs_order:
    r = df_pair[(df_pair["country_1"] == a) & (df_pair["country_2"] == b)]
    if r.empty:
        r = df_pair[(df_pair["country_1"] == b) & (df_pair["country_2"] == a)]
        if not r.empty:
            r = r.copy()
            r["cliffs_delta"] = -r["cliffs_delta"]  # flip if reversed
    if not r.empty:
        rows.append({
            "pair": f"{a} vs {b}",
            "delta": float(r["cliffs_delta"].iloc[0]),
            "p_holm": float(r["p_holm"].iloc[0])
        })
df_bar = pd.DataFrame(rows)

def star(p):
    if p < 1e-3: return "***"
    elif p < 1e-2: return "**"
    elif p < 5e-2: return "*"
    return "ns"

plt.figure(figsize=(8, 4))
x = np.arange(len(df_bar))
vals = df_bar["delta"].values
plt.bar(x, vals)
plt.axhline(0, ls="--", lw=1, alpha=0.6)

# dynamic headroom
dmax = float(np.max(np.abs(vals))) if len(vals) else 0.0
pad  = max(0.05, 0.20 * dmax)
ymin = 0 if np.all(vals >= 0) else -(dmax + pad)
ymax = dmax + pad
plt.ylim(ymin, ymax)


for i, (d, p) in enumerate(zip(vals, df_bar["p_holm"])):
    s = "***" if p < 1e-3 else ("**" if p < 1e-2 else ("*" if p < 5e-2 else "ns"))
    ytext = d + (0.02 if d >= 0 else -0.04)
    plt.text(i, ytext, f"{d:.3f}\n{s}", ha="center",
             va="bottom" if d >= 0 else "top")

plt.xticks(x, df_bar["pair"].tolist())
plt.ylabel("Cliff’s δ")
plt.title("Pairwise country differences (Cliff’s δ, Holm-adjusted p)")
plt.tight_layout()
plt.savefig(FIG_DIR / "cliffs_delta_pairs.png", dpi=FIG_DPI, bbox_inches="tight")
plt.savefig(FIG_DIR / "cliffs_delta_pairs.pdf", dpi=FIG_DPI, bbox_inches="tight", transparent=True)
plt.close()

captions.append({
    "figure": "cliffs_delta_pairs.pdf",
    "caption": "Pairwise effect sizes (Cliff’s δ) for country-level sentiment differences. "
               "Stars denote Holm-adjusted significance (* p<0.05, ** p<0.01, *** p<0.001)."
})

# --- D) Save captions for thesis use ---
cap_path = CAP_DIR / "figure_captions_visual_report.csv"
pd.DataFrame(captions).to_csv(cap_path, index=False, encoding="utf-8")

print("Saved publication-grade figures (PNG + PDF) to:", FIG_DIR)
print("Saved caption table:", cap_path)


Saved publication-grade figures (PNG + PDF) to: /content/Project_Dissertation/artifacts/figures
Saved caption table: /content/Project_Dissertation/artifacts/tables/figure_captions_visual_report.csv


# Cell — Export NLP Artifacts

In [ ]:
# ==============================================================
# Cell — Export NLP Artifacts (figures, tables, models, processed data)
# --------------------------------------------------------------
# Packages all NLP outputs from /content/Project_Dissertation/
# into a single ZIP file for download.
# ==============================================================

import shutil
from pathlib import Path
from datetime import datetime
from google.colab import files

ROOT_NLP = Path("/content/Project_Dissertation")
EXPORT_NLP = Path("/content/EXPORT_NLP")
EXPORT_NLP.mkdir(parents=True, exist_ok=True)

folders_to_copy = [
    ROOT_NLP / "artifacts",        # figures, tables, models, logs
    ROOT_NLP / "data" / "processed"  # parquet data
]

for folder in folders_to_copy:
    if folder.exists():
        dest = EXPORT_NLP / folder.name
        shutil.copytree(folder, dest, dirs_exist_ok=True)
        print(f"Copied: {folder} -> {dest}")
    else:
        print(f"[skip] {folder} not found.")

ts = datetime.now().strftime("%Y%m%d_%H%M")
zip_path_nlp = f"/content/NLP_Artifacts_{ts}.zip"
shutil.make_archive(zip_path_nlp.replace(".zip",""), 'zip', EXPORT_NLP)

print("\n✅ NLP artifacts archived successfully!")
print("📦 ZIP file created at:", zip_path_nlp)
print("\nDownload starting...")

files.download(zip_path_nlp)


Copied: /content/Project_Dissertation/artifacts -> /content/EXPORT_NLP/artifacts
Copied: /content/Project_Dissertation/data/processed -> /content/EXPORT_NLP/processed

✅ NLP artifacts archived successfully!
📦 ZIP file created at: /content/NLP_Artifacts_20260129_0952.zip

Download starting...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>